# Tugas Besar 2 IF3170 Inteligensi Artifisial

Prediksi pembatalan pemesanan penginapan.

## Group G02 - nutellabear

- Samuelson Dharmawan Tanuraharja (13524001)
- Edward David Rumahorbo (13524036)
- Narendra Dharma Wistara Marpaung (13524044)
- Reinhard Alfonzo Hutabarat (13524056)
- Miguel Rangga Deardo Sinaga (13524069)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


# Introduction <a name="1"></a>

Pembatalan pemesanan memengaruhi jumlah kamar yang perlu disiapkan. Model memperkirakan apakah sebuah pemesanan akan dibatalkan berdasarkan informasi pemesanan yang tersedia.

## Overview

Masalah ini merupakan klasifikasi biner. Target `is_canceled` bernilai 1 untuk pemesanan yang dibatalkan dan 0 untuk pemesanan yang tidak dibatalkan.

## Aim

Kami membandingkan lima model pada data validasi yang sama, memeriksa jenis kesalahan yang mereka buat, dan menghasilkan berkas prediksi dengan format yang sesuai untuk Kaggle.

## Metric

**Akurasi** adalah proporsi prediksi yang benar dan menjadi kriteria pemilihan konfigurasi pada jalur pengumpulan notebook ini. **Presisi** mengukur berapa banyak prediksi batal yang benar, sedangkan **recall** mengukur berapa banyak pembatalan aktual yang terdeteksi. **F1 kelas batal** adalah rata-rata harmonik presisi dan recall kelas 1. **F1 makro** merata-ratakan F1 kelas 0 dan 1 dengan bobot yang sama. **ROC-AUC** mengukur kemampuan model mengurutkan pemesanan batal di atas pemesanan yang tidak batal tanpa menetapkan ambang.

Metrik berbasis label di notebook menggunakan ambang 0,5. Skor publik Kaggle dilaporkan secara terpisah dari metrik validasi lokal.

## Dataset

`train.csv` memuat 27.373 pemesanan berlabel dan `test.csv` memuat 6.844 pemesanan tanpa label. Terdapat 17 fitur awal, termasuk jumlah tamu, lama menginap, tarif, dan kanal pemesanan. Kolom `id` tetap disimpan sebagai pengenal untuk berkas pengumpulan. Sekitar 32,77% pemesanan pada data latih berlabel batal.

## Approach

Kami membagi data berlabel menjadi data latih dan validasi secara berstrata. Statistik prapemrosesan dipelajari dari bagian latih. KNN, pohon keputusan C4.5, regresi logistik, SVM, dan Naive Bayes diimplementasikan dari awal, lalu hasilnya dibandingkan dengan model pembanding scikit-learn.

# Initialization <a name="2"></a>

Notebook dapat dijalankan dari akar repositori, dari folder `nb/`, atau di Kaggle dengan dataset kompetisi terpasang.

## Environment Setup

Untuk eksekusi lokal, pasang dependensi pada `requirements.txt` serta `ipykernel` di lingkungan Python yang dipilih sebagai kernel notebook. Di Kaggle, periksa ketersediaan dependensi sebelum menjalankan semua sel.

In [ ]:
# Pasang dependensi proyek pada kernel yang dipilih sebelum menjalankan notebook.

## Import Libraries

NumPy digunakan dalam implementasi model. Pandas mengelola data, Matplotlib membuat grafik, dan scikit-learn dipakai untuk prapemrosesan serta model pembanding.

In [ ]:
import os
import pickle
import random
import shutil
import time
from dataclasses import dataclass, field
from numbers import Integral, Real
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import KBinsDiscretizer, OrdinalEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## Settings

Tetapkan `IF3170_DATA_DIR` jika dataset disimpan di luar lokasi yang terdeteksi otomatis. Ketiga berkas dataset harus berada dalam satu folder. Keluaran ditulis ke `output/` di akar repositori atau `/kaggle/working/output` di Kaggle. Gunakan `IF3170_OUTPUT_DIR` untuk mengarahkan keluaran uji ke folder terpisah.

In [ ]:
def find_data_dir():
    required = ('train.csv', 'test.csv', 'sample_submission.csv')

    def has_dataset(folder):
        return all((folder / filename).is_file() for filename in required)

    if override := os.environ.get('IF3170_DATA_DIR'):
        folder = Path(override).expanduser().resolve()
        if not has_dataset(folder):
            raise FileNotFoundError(f'Dataset tidak lengkap di IF3170_DATA_DIR={folder}')
        return folder

    cwd = Path.cwd()
    for base in (cwd, *cwd.parents):
        for folder in (base / 'T2-IF3170-2026-2027', base / 'data', base):
            if has_dataset(folder):
                return folder

    kaggle_input = Path('/kaggle/input')
    matches = sorted({path.parent for path in kaggle_input.rglob('train.csv')
                      if has_dataset(path.parent)})
    if len(matches) == 1:
        return matches[0]
    if matches:
        raise ValueError(f'Beberapa dataset ditemukan: {matches}. Atur IF3170_DATA_DIR.')
    raise FileNotFoundError('Dataset tidak ditemukan. Tambahkan input Kaggle atau atur IF3170_DATA_DIR.')


class Settings:
    SEED = 42
    DATA_DIR = find_data_dir()
    PROJECT_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                        if (p / 'requirements.txt').is_file() and (p / 'nb').is_dir()),
                       Path.cwd())
    KAGGLE_WORKING = Path('/kaggle/working')
    OUTPUT_DIR = Path(os.environ.get('IF3170_OUTPUT_DIR') or
                      ((KAGGLE_WORKING if KAGGLE_WORKING.is_dir() else PROJECT_DIR) / 'output'))


CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data: {CFG.DATA_DIR}')
print(f'Keluaran: {CFG.OUTPUT_DIR}')

## Seed Everything

Nilai `seed=42` digunakan agar pembagian data dan proses acak dapat diulang pada lingkungan yang sama.

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

Kolom `id` dibaca sebagai teks agar representasinya tetap sama dengan berkas contoh pengumpulan.

In [ ]:
train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv', dtype={'id': str})
test_df  = pd.read_csv(CFG.DATA_DIR / 'test.csv', dtype={'id': str})

print(f'Train: {train_df.shape}  Test: {test_df.shape}')

# Data Understanding (Optional) <a name="3"></a>

Eksplorasi ini memeriksa tipe fitur, distribusi kelas, dan kualitas data sebelum pemodelan.

In [ ]:
TARGET = "is_canceled"
ID_COL = "id"
CATEGORICAL = ["meal_package", "room_category", "booking_channel"]

def find_invalid_rows(df):
    nights = df["weekend_nights"] + df["weekday_nights"]
    return {
        "zero_nights": nights == 0,
        "no_guests": (df["num_adults"] == 0) & (df["num_children"] == 0),
    }

def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())

def describe_data(df):
    feats = df.drop(columns=[c for c in (ID_COL,) if c in df.columns])
    info = pd.DataFrame({
        "dtype": feats.dtypes.astype(str),
        "n_missing": feats.isna().sum(),
        "n_unique": feats.nunique(),
    })
    print(f"Ukuran: {df.shape}")
    print(info.to_string())
    print(feats.describe().T.round(2).to_string())
    for c in CATEGORICAL:
        print(f"\n{c}")
        print(feats[c].value_counts(dropna=False).to_string())
    return info

def verify_quality(df, name="train"):
    feats = df.drop(columns=[c for c in (ID_COL, TARGET) if c in df.columns])
    n = len(df)

    miss = df.isna().sum()
    miss = miss[miss > 0]
    report = {
        "n_rows": n,
        "missing": pd.DataFrame({"n": miss, "pct": (miss / n * 100).round(2)}),
        "rows_with_any_missing": int(df.isna().any(axis=1).sum()),
        "duplicates_extra_copies": int(feats.duplicated().sum()),
    }

    dup_mask = feats.duplicated(keep=False)
    if TARGET in df.columns and dup_mask.any():
        grp = df[dup_mask].groupby(list(feats.columns), dropna=False)[TARGET].nunique()
        report["duplicate_groups_conflicting_label"] = int((grp > 1).sum())

    report["invalid"] = {k: int(v.sum()) for k, v in find_invalid_rows(df).items()}

    num = feats.select_dtypes("number")
    report["iqr_outliers"] = {
        c: iqr_outliers(num[c].dropna()) for c in num.columns if num[c].nunique() > 5
    }

    if TARGET in df.columns:
        report["class_share"] = df[TARGET].value_counts(normalize=True).round(4).to_dict()

    print(f"=== Kualitas data: {name} ({n} baris) ===")
    print(report["missing"].to_string())
    print(f"Baris dengan >=1 missing: {report['rows_with_any_missing']}")
    print(
        f"Duplikat tanpa id: {report['duplicates_extra_copies']} salinan berlebih, "
        f"grup label konflik: {report.get('duplicate_groups_conflicting_label', 'n/a')}"
    )
    print(f"Nilai tidak valid: {report['invalid']}")
    print(f"Outlier IQR: {report['iqr_outliers']}")
    if "class_share" in report:
        print(f"Proporsi kelas: {report['class_share']}")
    return report

def finish(fig, name, save_dir):
    fig.tight_layout()
    if save_dir is not None:
        Path(save_dir).mkdir(parents=True, exist_ok=True)
        fig.savefig(Path(save_dir) / f"{name}.png", dpi=150)
    plt.show()

def explore_data(df, save_dir=None):
    num_cols = [c for c in df.select_dtypes("number").columns if c != TARGET]

    fig, ax = plt.subplots(figsize=(4, 4))
    df[TARGET].value_counts(normalize=True).sort_index().plot.bar(ax=ax)
    ax.set_title("Proporsi kelas")
    finish(fig, "target_distribution", save_dir)

    fig, axes = plt.subplots(1, len(CATEGORICAL), figsize=(5 * len(CATEGORICAL), 4))
    for ax, c in zip(axes, CATEGORICAL):
        df.groupby(c, dropna=False)[TARGET].mean().sort_values().plot.barh(ax=ax)
        ax.axvline(df[TARGET].mean(), color="red", linestyle="--")
        ax.set_title(f"Rasio batal per {c}")
    finish(fig, "cancel_rate_categorical", save_dir)

    ncols = 4
    nrows = int(np.ceil(len(num_cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
    for ax, c in zip(axes.ravel(), num_cols):
        ax.hist(df[c].dropna(), bins=30)
        ax.set_title(c)
    for ax in axes.ravel()[len(num_cols):]:
        ax.axis("off")
    finish(fig, "numeric_distributions", save_dir)

    corr = df[num_cols + [TARGET]].corr()
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr)))
    ax.set_xticklabels(corr.columns, rotation=90)
    ax.set_yticks(range(len(corr)))
    ax.set_yticklabels(corr.columns)
    fig.colorbar(im, ax=ax)
    ax.set_title("Korelasi")
    finish(fig, "correlation", save_dir)

    return corr[TARGET].drop(TARGET).sort_values(key=np.abs, ascending=False)

def ks_statistic(a, b):
    a, b = np.sort(np.asarray(a, dtype=float)), np.sort(np.asarray(b, dtype=float))
    grid = np.concatenate([a, b])
    cdf_a = np.searchsorted(a, grid, side="right") / len(a)
    cdf_b = np.searchsorted(b, grid, side="right") / len(b)
    return float(np.abs(cdf_a - cdf_b).max())

def compare_train_test(train, test):
    cols = [c for c in train.columns if c not in (ID_COL, TARGET) and c in test.columns]
    rows = []
    for c in cols:
        tr, te = train[c], test[c]
        row = {"feature": c, "miss_train_pct": round(tr.isna().mean() * 100, 2),
               "miss_test_pct": round(te.isna().mean() * 100, 2)}
        if pd.api.types.is_numeric_dtype(tr):
            a, b = tr.dropna(), te.dropna()
            row.update(kind="numeric", distance=round(ks_statistic(a, b), 4),
                       mean_shift_std=round((b.mean() - a.mean()) / (a.std() or 1.0), 4))
        else:
            p = tr.value_counts(normalize=True)
            q = te.value_counts(normalize=True)
            levels = p.index.union(q.index)
            tvd = 0.5 * (p.reindex(levels, fill_value=0) - q.reindex(levels, fill_value=0)).abs().sum()
            row.update(kind="categorical", distance=round(float(tvd), 4),
                       new_in_test=sorted(map(str, q.index.difference(p.index))))
        rows.append(row)
    out = pd.DataFrame(rows).set_index("feature")
    print(f"=== Train ({len(train)}) vs test ({len(test)}) ===")
    print(out.to_string())
    return out

## Describe Data

Tabel merangkum tipe, nilai hilang, jumlah nilai unik, dan statistik deskriptif.

In [ ]:
eda_describe = describe_data(train_df)

### Insights

`meal_package`, `room_category`, dan `booking_channel` merupakan fitur kategorikal. `num_children` memiliki 3.035 nilai hilang dan `nightly_rate_idr` memiliki 2.038 nilai hilang pada data latih.

## Explore Data

Grafik menunjukkan distribusi fitur dan variasi proporsi pembatalan antarkategori.

In [ ]:
eda_target_corr = explore_data(train_df, CFG.OUTPUT_DIR / 'eda')
eda_target_corr.head(10)

### Insights

Proporsi pembatalan pada data latih sekitar 32,77%. Pada data validasi, aturan yang selalu memprediksi kelas tidak batal mencapai akurasi sekitar 0,6723. Nilai ini menjadi pembanding sederhana untuk akurasi model.

### Train Versus Test

Data uji diperiksa untuk melihat pergeseran distribusi. Statistik dari pemeriksaan ini tidak digunakan untuk melatih model.

In [ ]:
eda_shift = compare_train_test(train_df, test_df)

### Insights

Pemeriksaan menemukan 10,91% nilai `num_children` hilang pada data uji, mendekati 11,09% pada data latih. Ketiga fitur kategorikal tidak memperlihatkan kategori baru pada data uji dalam pemeriksaan ini.

## Verify Data Quality

Pemeriksaan mencakup nilai hilang, baris berulang, label yang bertentangan, dan nilai yang melanggar aturan data.

In [ ]:
eda_quality_train = verify_quality(train_df, 'train')
eda_quality_test = verify_quality(test_df, 'test')

### Insights

Sebanyak 10.737 baris latih memiliki sedikitnya satu nilai hilang. Pemeriksaan data latih sebelum pembagian menemukan 66 salinan duplikat berdasarkan fitur, dua kelompok dengan label bertentangan, dan 64 baris tanpa malam menginap.

# Training and Validation Split <a name="4"></a>

Pembagian berstrata dengan `seed=42` menghasilkan 21.898 baris latih sebelum pembersihan dan 5.475 baris validasi. Data uji digunakan untuk membuat prediksi pengumpulan dan pemeriksaan distribusi.

In [ ]:
"""Preprocessing: split, cleaning, langkah persiapan, dan pipeline. Semua `fit` hanya pada train."""

TARGET = "is_canceled"
ID_COL = "id"
CATEGORICAL = ["meal_package", "room_category", "booking_channel"]


def feature_cols(df):
    return [c for c in df.columns if c not in (ID_COL, TARGET)]


def split_train_val(df, val_size=0.2, seed=42):
    """Split train.csv berstratifikasi pada target."""
    return train_test_split(df, test_size=val_size, random_state=seed, stratify=df[TARGET])


def drop_duplicates_and_invalid(df):
    """Buang duplikat dan baris invalid (hanya untuk train). Kembalikan (df_bersih, laporan)."""
    feats = feature_cols(df)
    report = {"rows_in": len(df)}

    # grup duplikat dengan label bertentangan dibuang seluruhnya, grup lain disisakan satu baris
    conflict = df.groupby(feats, dropna=False)[TARGET].transform("nunique") > 1
    report["conflicting_duplicates"] = int(conflict.sum())
    df = df[~conflict]
    dup = df.duplicated(subset=feats)
    report["duplicates"] = int(dup.sum())
    df = df[~dup]

    # malam = 0, tarif = 0, dan dewasa = 0 sengaja dipertahankan (alasan di notebook)
    num = df[feats].select_dtypes("number")
    invalid = (df["num_adults"].eq(0) & df["num_children"].eq(0)) | (num < 0).any(axis=1)
    report["invalid"] = int(invalid.sum())
    df = df[~invalid]

    report["rows_out"] = len(df)
    return df, report


def construct_features(df):
    """Tambah total_nights, total_guests, nightly_rate_per_guest."""
    df = df.copy()
    df["total_nights"] = df["weekend_nights"] + df["weekday_nights"]
    df["total_guests"] = df["num_adults"] + df["num_children"]
    df["nightly_rate_per_guest"] = df["nightly_rate_idr"] / df["total_guests"].clip(lower=1)
    return df


class Imputer:
    """Isi NaN: median (numerik), modus (kategorikal). Kolom di indicator_cols diberi <nama>_missing."""

    def __init__(self, indicator_cols=("special_requests",)):
        self.indicator_cols = list(indicator_cols)

    def fit(self, df):
        cols = feature_cols(df)
        num = [c for c in cols if c not in CATEGORICAL]
        self.fill_ = {**df[num].median().to_dict(),
                      **{c: df[c].mode()[0] for c in cols if c in CATEGORICAL}}
        return self

    def transform(self, df):
        df = df.copy()
        for c in self.indicator_cols:
            df[f"{c}_missing"] = df[c].isna().astype(int)
        return df.fillna(self.fill_)


class Clipper:
    """Potong kolom kontinu ke kuantil [lower, upper] train."""

    # kolom hitungan (mis. prior_cancellations) tidak dipotong: kuantil 99%-nya 0, sinyalnya akan hilang
    def __init__(self, cols=("days_before_arrival", "nightly_rate_idr", "nightly_rate_per_guest"),
                 lower=0.01, upper=0.99):
        self.cols, self.lower, self.upper = list(cols), lower, upper

    def fit(self, df):
        self.bounds_ = df[self.cols].quantile([self.lower, self.upper])
        return self

    def transform(self, df):
        df = df.copy()
        for c in self.bounds_.columns:
            df[c] = df[c].clip(*self.bounds_[c])
        return df


class CategoricalEncoder:
    """One-hot dengan kategori dari train. Kategori tak dikenal menjadi semua 0."""

    def fit(self, df):
        self.categories_ = {c: sorted(df[c].dropna().unique()) for c in CATEGORICAL}
        return self

    def transform(self, df):
        df = df.copy()
        for c, cats in self.categories_.items():
            for v in cats:
                df[f"{c}={v}"] = (df[c] == v).astype(int)
        return df.drop(columns=list(self.categories_))


class Scaler:
    """Standarisasi ("standard") atau min-max ("minmax") dengan statistik train."""

    def __init__(self, method="standard"):
        self.method = method

    def fit(self, df):
        X = df[feature_cols(df)]
        if self.method == "standard":
            self.shift_, scale = X.mean(), X.std(ddof=0)
        else:
            self.shift_, scale = X.min(), X.max() - X.min()
        self.scale_ = scale.replace(0, 1)  # kolom konstan: hindari bagi nol
        return self

    def transform(self, df):
        df = df.copy()
        cols = list(self.shift_.index)
        df[cols] = (df[cols] - self.shift_) / self.scale_
        return df


class FeatureSelector:
    """Pilih k fitur dengan mutual information tertinggi terhadap target. Skor di `scores_`."""

    def __init__(self, k=None, min_score=0.0, seed=42):
        self.k, self.min_score, self.seed = k, min_score, seed

    def fit(self, df):
        cols = feature_cols(df)
        mi = mutual_info_classif(df[cols], df[TARGET], random_state=self.seed)
        self.scores_ = pd.Series(mi, index=cols).sort_values(ascending=False)
        keep = self.scores_[self.scores_ >= self.min_score]
        self.selected_ = list(keep.index[:self.k] if self.k else keep.index)
        return self

    def transform(self, df):
        return df[[c for c in df.columns if c in self.selected_ or c in (ID_COL, TARGET)]]


class DimReducer:
    """PCA. n_components < 1 berarti proporsi varians yang dipertahankan."""

    def __init__(self, n_components=0.95, seed=42):
        self.n_components, self.seed = n_components, seed

    def fit(self, df):
        self.pca_ = PCA(self.n_components, random_state=self.seed).fit(df[feature_cols(df)])
        return self

    def transform(self, df):
        Z = self.pca_.transform(df[feature_cols(df)])
        out = pd.DataFrame(Z, index=df.index, columns=[f"pc{i + 1}" for i in range(Z.shape[1])])
        return out.join(df[[c for c in (ID_COL, TARGET) if c in df.columns]])


def handle_imbalance(X, y, seed=42):
    """Random oversampling kelas minoritas sampai seimbang. Hanya untuk train."""
    X, y = np.asarray(X), np.asarray(y)
    counts = np.bincount(y)
    minority = counts.argmin()
    extra = np.random.default_rng(seed).choice(np.flatnonzero(y == minority), counts.max() - counts.min())
    idx = np.r_[np.arange(len(y)), extra]
    return X[idx], y[idx]


class PreparationPipeline:
    """Jalankan langkah berurutan. transform mengembalikan (X, y); y = None bila tidak ada target."""

    def __init__(self, steps):
        self.steps = steps

    def _run(self, df, fit):
        df = df.drop(columns=[ID_COL], errors="ignore")
        for step in self.steps:
            if callable(step):  # fungsi tanpa state, mis. construct_features
                df = step(df)
            else:
                df = (step.fit(df) if fit else step).transform(df)
        return df

    def fit(self, df):
        self.columns_ = feature_cols(self._run(df, fit=True))
        return self

    def transform(self, df):
        out = self._run(df, fit=False)
        y = out[TARGET].to_numpy().astype(int) if TARGET in out else None
        return out[self.columns_], y

    def fit_transform(self, df):
        return self.fit(df).transform(df)


def numeric_pipeline(select_k=None, pca=None):
    """Tampilan numerik (KNN, LR, SVM): tanpa NaN, one-hot, terskala."""
    steps = [construct_features, Imputer(), Clipper(), CategoricalEncoder(), Scaler()]
    if select_k:
        steps.append(FeatureSelector(k=select_k))
    if pca:
        steps.append(DimReducer(pca))
    return PreparationPipeline(steps)


def tree_pipeline(impute=False):
    """Tampilan pohon (DTL, NB): kategori tetap teks, NaN dibiarkan kecuali impute=True."""
    return PreparationPipeline([construct_features] + ([Imputer()] if impute else []))


def prepare(train_df, val_df, view="numeric", **kw):
    """Bersihkan train, fit pipeline, transform keduanya. Untuk evaluation.cross_validate."""
    train_df, _ = drop_duplicates_and_invalid(train_df)
    pipe = (numeric_pipeline if view == "numeric" else tree_pipeline)(**kw).fit(train_df)
    return (*pipe.transform(train_df), *pipe.transform(val_df))


def make_views(train_df, *others):
    """Fit kedua pipeline pada train, transform semua df. Kembalikan {"numeric": [(X, y), ...], "tree": [...]}."""
    out = {}
    for name, pipe in [("numeric", numeric_pipeline()), ("tree", tree_pipeline())]:
        pipe.fit(train_df)
        out[name] = [pipe.transform(d) for d in (train_df, *others)]
    return out

In [ ]:
train_set, val_set = split_train_val(train_df, val_size=0.2, seed=CFG.SEED)

print(f'Train: {train_set.shape}  Validation: {val_set.shape}')
pd.DataFrame({
    'train': train_set[TARGET].value_counts(normalize=True),
    'validation': val_set[TARGET].value_counts(normalize=True),
}).round(4)

# Data Preparation <a name="5"></a>

Data latih dibersihkan terlebih dahulu. Transformasi berikutnya mempertahankan urutan baris pada data validasi dan uji.

Imputasi, batas pemotongan nilai, pengodean, dan penskalaan dipelajari dari bagian latih. Pada validasi silang, setiap lipatan melatih ulang tahap persiapan dari bagian latih lipatan tersebut.

## Select Data

Model menggunakan fitur pemesanan. Kolom `id` berfungsi sebagai pengenal keluaran, sedangkan `is_canceled` menjadi target.

In [ ]:
selected_attributes = feature_cols(train_set)

print(f'{len(selected_attributes)} attributes: {selected_attributes}')
print(f'Categorical: {CATEGORICAL}')

## Clean Data

### Missing Values

Alur numerik mengisi fitur numerik dengan median dan kategori dengan modus. Alur pohon mempertahankan nilai hilang agar C4.5 dapat membaginya dengan bobot pecahan.

In [ ]:
missing_cols = [c for c in selected_attributes if train_set[c].isna().any()]
missing_table = pd.DataFrame({
    'train': train_set[missing_cols].isna().mean(),
    'validation': val_set[missing_cols].isna().mean(),
    'test': test_df[missing_cols].isna().mean(),
    'cancel rate if missing': [train_set.loc[train_set[c].isna(), TARGET].mean() for c in missing_cols],
    'cancel rate if present': [train_set.loc[train_set[c].notna(), TARGET].mean() for c in missing_cols],
}).round(4)

imputer = Imputer().fit(train_set)
imputed_val = imputer.transform(val_set)
print('Fill values from train:', {c: imputer.fill_[c] for c in missing_cols})
print('Missing cells in validation after imputation:', int(imputed_val[selected_attributes].isna().sum().sum()))
missing_table

### Duplicates

Pembersihan pada bagian latih menangani baris berulang dan kelompok dengan label yang bertentangan. Sebanyak 21 baris validasi memiliki kombinasi fitur dan label yang juga muncul pada bagian latih bersih. Tumpang tindih ini perlu diperhitungkan saat menafsirkan skor pada pembagian ini.

In [ ]:
train_clean, clean_report = drop_duplicates_and_invalid(train_set)

cross_split_copies = val_set.merge(train_clean[selected_attributes + [TARGET]].drop_duplicates(),
                                   on=selected_attributes + [TARGET], how='inner')
print(f'Validation rows identical to a training row: {len(cross_split_copies)}')
pd.Series(clean_report, name='train rows')

### Outliers

Alur numerik membatasi nilai ekstrem pada fitur yang dipilih menurut persentil data latih.

In [ ]:
constructed_train = construct_features(train_clean)
imputed_train = Imputer().fit(constructed_train).transform(constructed_train)
clipper = Clipper().fit(imputed_train)

outlier_table = clipper.bounds_.T.set_axis(['lower (1%)', 'upper (99%)'], axis=1)
outlier_table['train rows clipped'] = [
    int((~imputed_train[c].between(lo, hi)).sum()) for c, (lo, hi) in outlier_table.iterrows()
]
display(outlier_table)

imputed_train.groupby(imputed_train['prior_cancellations'] > 0)[TARGET].agg(['size', 'mean'])

### Data Errors

Label diperiksa agar hanya berisi 0 dan 1. Daftar kategori yang tampil berasal dari bagian latih.

In [ ]:
numeric_attributes = [c for c in selected_attributes if c not in CATEGORICAL]
checks = {
    'zero nights': train_set['weekend_nights'] + train_set['weekday_nights'] == 0,
    'zero guests': train_set['num_adults'].eq(0) & train_set['num_children'].eq(0),
    'zero adults': train_set['num_adults'].eq(0),
    'zero nightly rate': train_set['nightly_rate_idr'].eq(0),
    'negative value': (train_set[numeric_attributes] < 0).any(axis=1),
}

print('Target values:', sorted(train_set[TARGET].unique()))
print('Categories:', {c: sorted(train_set[c].dropna().unique()) for c in CATEGORICAL})
pd.DataFrame({
    'train rows': {name: int(mask.sum()) for name, mask in checks.items()},
    'cancel rate': {name: train_set.loc[mask, TARGET].mean() for name, mask in checks.items()},
}).round(4)

## Construct Data

### Attribute Construction

<!-- rubric v3: derived columns, rate_per_guest_night -->

Tiga fitur turunan merangkum jumlah malam (`total_nights`), jumlah tamu (`total_guests`), dan tarif per tamu (`nightly_rate_per_guest`).

In [ ]:
derived_attributes = ['total_nights', 'total_guests', 'nightly_rate_per_guest']

construction_table = constructed_train[derived_attributes].describe().T
construction_table['spearman with target'] = [
    constructed_train[c].corr(constructed_train[TARGET], method='spearman') for c in derived_attributes
]
construction_table.round(3)

### Feature Scaling

Alur numerik melakukan standardisasi setelah imputasi, pembatasan nilai ekstrem, dan pengodean. Alur pohon menggunakan fitur tanpa standardisasi.

In [ ]:
unscaled_prep = PreparationPipeline([construct_features, Imputer(), Clipper(), CategoricalEncoder()]).fit(train_clean)
unscaled_train, _ = unscaled_prep.transform(train_clean)
unscaled_val, _ = unscaled_prep.transform(val_set)

standard = Scaler('standard').fit(unscaled_train)
minmax = Scaler('minmax').fit(unscaled_train)
pd.DataFrame({
    'mean before': unscaled_train.mean(),
    'std before': unscaled_train.std(ddof=0),
    'mean standard (val)': standard.transform(unscaled_val).mean(),
    'std standard (val)': standard.transform(unscaled_val).std(ddof=0),
    'min min-max (val)': minmax.transform(unscaled_val).min(),
    'max min-max (val)': minmax.transform(unscaled_val).max(),
}).round(3)

### Encoding

Alur numerik memakai pengodean *one-hot*. Alur pohon mempertahankan kategori asli untuk split berdasarkan kategori.

In [ ]:
encoder = CategoricalEncoder().fit(train_clean)
encoded_sample = encoder.transform(val_set.head())

print('Categories per attribute (from train):', {c: len(v) for c, v in encoder.categories_.items()})
pd.concat([val_set.head()[CATEGORICAL], encoded_sample.filter(like='=')], axis=1)

### Dimensionality Reduction

Eksperimen KNN membandingkan pemilihan 20 fitur melalui informasi mutual dan transformasi PCA dengan alur dasar. PCA mempertahankan 25 dari 34 kolom untuk melampaui 95% ragam kumulatif pada eksekusi lokal.

In [ ]:
mi_selector = numeric_pipeline(select_k=20).fit(train_clean).steps[-1]
pca_check = numeric_pipeline(pca=0.95).fit(train_clean).steps[-1].pca_

print(f'PCA keeps {pca_check.n_components_} of {pca_check.n_features_in_} columns for 95% of the variance')
display(pd.Series(np.cumsum(pca_check.explained_variance_ratio_), index=range(1, pca_check.n_components_ + 1),
                  name='cumulative explained variance').round(4))
mi_selector.scores_.round(4).to_frame('mutual information with target')

## Class Balance

Penambahan salinan sampel kelas minoritas diuji pada bagian latih KNN. Data validasi mempertahankan distribusi kelas aslinya.

In [ ]:
balance_X, balance_y = numeric_pipeline().fit_transform(train_clean)
_, oversampled_y = handle_imbalance(balance_X, balance_y, seed=CFG.SEED)

pd.DataFrame({
    'train': train_set[TARGET].value_counts(),
    'train after cleaning': train_clean[TARGET].value_counts(),
    'train after oversampling': pd.Series(oversampled_y).value_counts(),
    'validation': val_set[TARGET].value_counts(),
})

## Format Data

KNN, regresi logistik, dan SVM memakai tampilan numerik. DTL memakai tampilan pohon dengan fitur turunan. Konfigurasi Naive Bayes yang digunakan untuk pengumpulan memakai fitur asli.

In [ ]:
views = make_views(train_clean, val_set, test_df)
(X_train, y_train), (X_val, y_val), (X_test, _) = views['numeric']
(T_train, _), (T_val, _), (T_test, _) = views['tree']
tree_categorical = [T_train.columns.get_loc(c) for c in CATEGORICAL]

pd.DataFrame({
    'numeric view (KNN, LR, SVM)': [X_train.shape, X_val.shape, X_test.shape,
                                    int(X_train.isna().sum().sum()), sorted(set(X_train.dtypes.astype(str)))],
    'tree view (DTL, NB)': [T_train.shape, T_val.shape, T_test.shape,
                            int(T_train.isna().sum().sum()), sorted(set(T_train.dtypes.astype(str)))],
}, index=['train', 'validation', 'test', 'missing cells in train', 'dtypes'])

# Preparation Pipeline <a name="6"></a>

<!-- rubric v3: fitted steps, fitted_prep_steps -->

Sel ini menampilkan tahap yang dipasang pada setiap alur. Model numerik dan pohon menggunakan representasi yang sesuai dengan cara masing-masing memproses fitur.

In [ ]:
for name, pipe in [('numeric view', numeric_pipeline()), ('tree view', tree_pipeline())]:
    print(f"{name}: {' -> '.join(getattr(s, '__name__', type(s).__name__) for s in pipe.steps)}")

assert (X_train.index == T_train.index).all() and (X_val.index == T_val.index).all()
assert len(X_test) == len(T_test) == len(test_df)


def prepare_numeric(train_part, val_part):
    return prepare(train_part, val_part, 'numeric')

# Modeling and Validation <a name="7"></a>

Lima keluarga model dievaluasi pada baris validasi yang sama. Tabel per model memperlihatkan konfigurasi yang diuji, sedangkan tabel gabungan menandai konfigurasi yang dipakai untuk menghasilkan berkas pengumpulan.

In [ ]:
"""Modul evaluasi bersama. Semua fungsi menerima y (0/1) dan y_pred (0/1) atau skor peluang."""


def confusion_matrix(y, y_pred):
    """[[TN, FP], [FN, TP]], urutan sama dengan sklearn.metrics.confusion_matrix."""
    y, y_pred = np.asarray(y).astype(int), np.asarray(y_pred).astype(int)
    return np.array([[np.sum((y == 0) & (y_pred == 0)), np.sum((y == 0) & (y_pred == 1))],
                     [np.sum((y == 1) & (y_pred == 0)), np.sum((y == 1) & (y_pred == 1))]])


def accuracy(y, y_pred):
    return float(np.mean(np.asarray(y) == np.asarray(y_pred)))


def precision(y, y_pred):
    (_, fp), (_, tp) = confusion_matrix(y, y_pred)
    return float(tp / (tp + fp)) if tp + fp else 0.0


def recall(y, y_pred):
    (_, _), (fn, tp) = confusion_matrix(y, y_pred)
    return float(tp / (tp + fn)) if tp + fn else 0.0


def f1(y, y_pred):
    p, r = precision(y, y_pred), recall(y, y_pred)
    return 2 * p * r / (p + r) if p + r else 0.0


def roc_auc(y, score):
    """AUC = peluang skor positif > skor negatif (statistik Mann-Whitney). Skor kembar diberi rata-rata peringkat."""
    y, score = np.asarray(y).astype(int), np.asarray(score, dtype=float)
    order = np.argsort(score, kind="mergesort")
    s = score[order]
    ranks = np.empty(len(s))
    # peringkat 1..n, lalu tiap kelompok skor sama diberi rata-rata peringkatnya
    starts = np.r_[0, np.flatnonzero(np.diff(s)) + 1]
    ends = np.r_[starts[1:], len(s)]
    for a, b in zip(starts, ends):
        ranks[a:b] = (a + 1 + b) / 2
    r = np.empty(len(s))
    r[order] = ranks
    n_pos, n_neg = y.sum(), len(y) - y.sum()
    return float((r[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def scores(y, proba, threshold=0.5):
    """Semua metrik dalam satu dict, dari peluang kelas 1."""
    y_pred = (np.asarray(proba) >= threshold).astype(int)
    return {"accuracy": accuracy(y, y_pred), "precision": precision(y, y_pred),
            "recall": recall(y, y_pred), "f1": f1(y, y_pred), "roc_auc": roc_auc(y, proba)}


def stratified_kfold(y, k=5, seed=42):
    """Daftar (idx_train, idx_val). Tiap kelas diacak lalu dibagi rata ke k fold, jadi proporsi kelas tiap fold sama."""
    y = np.asarray(y)
    rng = np.random.default_rng(seed)
    fold_of = np.empty(len(y), dtype=int)
    for c in np.unique(y):
        idx = rng.permutation(np.flatnonzero(y == c))
        fold_of[idx] = np.arange(len(idx)) % k
    return [(np.flatnonzero(fold_of != f), np.flatnonzero(fold_of == f)) for f in range(k)]


def cross_validate(model_fn, prepare, df, k=5, seed=42, target="is_canceled"):
    """Stratified k-fold. `prepare(train_df, val_df) -> (X_tr, y_tr, X_va, y_va)` dipanggil per fold,
    jadi preprocessing di-fit hanya pada bagian train fold itu (tanpa leakage). Kembalikan metrik per fold."""
    rows = []
    for f, (tr, va) in enumerate(stratified_kfold(df[target].to_numpy(), k, seed)):
        X_tr, y_tr, X_va, y_va = prepare(df.iloc[tr], df.iloc[va])
        proba = model_fn().fit(X_tr, y_tr).predict_proba(X_va)
        rows.append({"fold": f, **scores(y_va, proba)})
    out = pd.DataFrame(rows).set_index("fold")
    out.loc["mean"] = out.mean()
    return out


def compare_with_sklearn(pairs, X_tr, y_tr, X_va, y_va):
    """pairs: {nama: (model_scratch, model_sklearn)}, keduanya belum di-fit. Satu baris per (model, versi)."""
    rows = []
    for name, (scratch, sk) in pairs.items():
        for impl, m in [("scratch", scratch), ("sklearn", sk)]:
            p = m.fit(X_tr, y_tr).predict_proba(X_va)
            p = p[:, 1] if p.ndim == 2 else p  # sklearn mengembalikan (m, 2), scratch (m,)
            rows.append({"model": name, "impl": impl, **scores(y_va, p)})
    return pd.DataFrame(rows).set_index(["model", "impl"])


def error_analysis(df_val, y, y_pred, cols=None):
    """Bandingkan rata-rata fitur (numerik) pada TP/TN/FP/FN. df_val berisi fitur mentah baris validasi."""
    y, y_pred = np.asarray(y), np.asarray(y_pred)
    kind = np.select([(y == 1) & (y_pred == 1), (y == 0) & (y_pred == 0), (y == 0) & (y_pred == 1)],
                     ["TP", "TN", "FP"], "FN")
    num = df_val[cols] if cols else df_val.select_dtypes("number")
    out = num.groupby(kind).mean().T
    out.loc["n_rows"] = pd.Series(kind).value_counts()
    return out

## K-Nearest Neighbors

<!-- rubric v3: pairwise distances, val_neighbor_dists -->

Eksperimen membandingkan empat nilai `k`, dua pembobotan, tiga metrik jarak, dan tiga alternatif prapemrosesan. Konfigurasi pengumpulan menggunakan `k=15`, jarak Manhattan, dan bobot berdasarkan jarak pada alur dasar. Run lokal menghasilkan akurasi 0,8490 dan F1 kelas batal 0,7573. Validasi silang lima lipatan menguji konfigurasi terpilih dengan melatih ulang alur pada setiap lipatan.

In [ ]:
"""KNN from scratch (NumPy).

Referensi: E. Ferdian, slide "KNN", IF3170 2026; P. Cunningham & S. J. Delany, "k-Nearest Neighbour Classifiers:
2nd Edition", arXiv:2004.04523, 2020. Penanganan jarak 0 pada weights="distance" mengikuti scikit-learn.
"""


class KNN:
    """metric: "euclidean", "manhattan", atau "minkowski" (pakai p). weights: "uniform" atau "distance"."""

    def __init__(self, k=5, metric="euclidean", p=2, weights="uniform", batch_size=256):
        self.k, self.metric, self.p, self.weights, self.batch_size = k, metric, p, weights, batch_size

    def fit(self, X, y):
        self.X_ = np.array(X, dtype=float)  # salinan, X pemanggil tidak diubah
        self.y_ = np.array(y).astype(int)
        return self

    def _pairwise_distances(self, Xq):
        """Matriks jarak (len(Xq), n_train)."""
        p = {"euclidean": 2, "manhattan": 1}.get(self.metric, self.p)
        if p == 2:  # ||a-b||^2 = ||a||^2 + ||b||^2 - 2ab, cukup satu perkalian matriks
            sq = (Xq ** 2).sum(1)[:, None] + (self.X_ ** 2).sum(1)[None, :] - 2 * Xq @ self.X_.T
            return np.sqrt(np.maximum(sq, 0))
        D = np.zeros((len(Xq), len(self.X_)))
        for j in range(Xq.shape[1]):  # per fitur, agar tidak membentuk array 3D (batch, n_train, d)
            D += np.abs(Xq[:, j, None] - self.X_[None, :, j]) ** p
        return D if p == 1 else D ** (1 / p)

    def kneighbors(self, X):
        """(jarak, indeks) k tetangga terdekat, terurut dari yang terdekat. Diproses per batch."""
        X = np.asarray(X, dtype=float)
        dist, idx = [], []
        for s in range(0, len(X), self.batch_size):
            D = self._pairwise_distances(X[s:s + self.batch_size])
            nn = np.argpartition(D, self.k - 1, axis=1)[:, :self.k]  # k terkecil, belum terurut
            d = np.take_along_axis(D, nn, 1)
            order = np.argsort(d, axis=1, kind="stable")
            dist.append(np.take_along_axis(d, order, 1))
            idx.append(np.take_along_axis(nn, order, 1))
        return np.vstack(dist), np.vstack(idx)

    def predict_proba(self, X):
        """Proporsi kelas 1 di antara k tetangga (berbobot 1/jarak bila weights="distance")."""
        dist, idx = self.kneighbors(X)
        labels = self.y_[idx]
        if self.weights == "uniform":
            return labels.mean(1)
        with np.errstate(divide="ignore"):
            w = 1 / dist
        exact = np.isinf(w)
        w[exact.any(1)] = exact[exact.any(1)]  # ada jarak 0: hanya titik identik yang dihitung
        return (w * labels).sum(1) / w.sum(1)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            return pickle.load(f)

In [ ]:
knn_grid = []
for k in [5, 15, 25, 41]:
    for weights in ['uniform', 'distance']:
        proba = KNN(k=k, weights=weights).fit(X_train, y_train).predict_proba(X_val)
        knn_grid.append({'k': k, 'weights': weights, **scores(y_val, proba)})
knn_grid = pd.DataFrame(knn_grid)
knn_grid.round(4)

In [ ]:
knn_best = knn_grid.loc[knn_grid['accuracy'].idxmax()]
knn_metrics = []
for metric, p in [('euclidean', 2), ('manhattan', 1), ('minkowski', 3)]:
    model = KNN(k=int(knn_best['k']), metric=metric, p=p, weights=knn_best['weights']).fit(X_train, y_train)
    knn_metrics.append({'metric': metric, 'p': p, **scores(y_val, model.predict_proba(X_val))})
knn_metrics = pd.DataFrame(knn_metrics)

best_metric = knn_metrics.loc[knn_metrics['accuracy'].idxmax()]
knn_params = {'k': int(knn_best['k']), 'metric': best_metric['metric'], 'p': int(best_metric['p']),
              'weights': knn_best['weights']}
print('Chosen:', knn_params)
knn_metrics.round(4)

In [ ]:
knn_model = KNN(**knn_params).fit(X_train, y_train)
knn_val_proba = knn_model.predict_proba(X_val)

knn_variants = {'default pipeline': scores(y_val, knn_val_proba)}
for name, pipe, oversample in [('top 20 features (mutual information)', numeric_pipeline(select_k=20), False),
                               ('PCA 95% variance', numeric_pipeline(pca=0.95), False),
                               ('random oversampling', numeric_pipeline(), True)]:
    pipe.fit(train_clean)
    X_a, y_a = pipe.transform(train_clean)
    X_b, y_b = pipe.transform(val_set)
    if oversample:
        X_a, y_a = handle_imbalance(X_a, y_a, seed=CFG.SEED)
    knn_variants[name] = scores(y_b, KNN(**knn_params).fit(X_a, y_a).predict_proba(X_b))
pd.DataFrame(knn_variants).T.round(4)

In [ ]:
knn_cv = cross_validate(lambda: KNN(**knn_params), prepare_numeric, train_df, k=5, seed=CFG.SEED)
knn_cv.round(4)

## Decision Tree Learning

Model memilih split berdasarkan *gain ratio*. Fitur numerik dipisahkan menggunakan ambang, sedangkan kategori dapat membentuk beberapa cabang. Nilai hilang dibagi ke cabang menurut proporsi berbobot. Pohon pada eksekusi lokal memiliki 4.634 simpul dan kedalaman 20. Pada data validasi, akurasinya 0,8449 dan F1 kelas batalnya 0,7530.

In [ ]:
def _entropy_counts(counts):
    """Entropy sepanjang sumbu terakhir; kelompok kosong bernilai nol."""
    counts = np.asarray(counts, dtype=float)
    total = counts.sum(axis=-1, keepdims=True)
    p = np.divide(counts, total, out=np.zeros_like(counts), where=total > 0)
    logs = np.zeros_like(p)
    np.log2(p, out=logs, where=p > 0)
    return -(p * logs).sum(axis=-1)


def _labels_weights(y, sample_weight=None):
    """Validasi label biner dan bobot sampel nonnegatif."""
    y = np.asarray(y)
    if y.ndim != 1 or not np.isin(y, [0, 1]).all():
        raise ValueError("y harus array satu dimensi berisi label 0/1.")
    weights = (np.ones(len(y)) if sample_weight is None
               else np.asarray(sample_weight, dtype=float))
    if (weights.shape != y.shape or not np.isfinite(weights).all()
            or (weights < 0).any()):
        raise ValueError("Bobot harus berukuran sama dengan y, finite, dan nonnegatif.")
    return y.astype(np.int64), weights


def entropy(y, sample_weight=None):
    """Entropy label biner, opsional dengan bobot pecahan; entropy([]) = 0."""
    y, weights = _labels_weights(y, sample_weight)
    return float(_entropy_counts(np.bincount(y, weights=weights, minlength=2)))


def _missing(column):
    """Tandai nilai None dan NaN sebagai nilai yang hilang."""
    if column.dtype.kind in "fci":
        return np.isnan(column)
    return np.fromiter((v is None or v != v for v in column),
                       dtype=bool, count=len(column))


def _partition_scores(counts, missing_weight):
    """Hitung gain terkoreksi dan split information dengan memperhitungkan bobot missing."""
    counts = np.asarray(counts, dtype=float)
    masses = counts.sum(axis=1)
    known = masses.sum()
    total = known + missing_weight
    if known <= 0 or total <= 0:
        return 0.0, 0.0
    gain_known = (_entropy_counts(counts.sum(axis=0))
                  - np.dot(masses / known, _entropy_counts(counts)))
    gain = max(0.0, float(known / total * gain_known))
    split_info = float(_entropy_counts(np.append(masses, missing_weight)))
    return gain, split_info


def information_gain(y, branches, sample_weight=None):
    """Gain split; None/NaN dalam branches menandakan atribut tidak diketahui."""
    y, weights = _labels_weights(y, sample_weight)
    branches = np.asarray(branches)
    if branches.shape != y.shape:
        raise ValueError("branches harus satu dimensi dengan panjang sama dengan y.")
    missing = _missing(branches)
    values, inverse = np.unique(branches[~missing], return_inverse=True)
    counts = np.zeros((len(values), 2))
    np.add.at(counts, (inverse, y[~missing]), weights[~missing])
    return _partition_scores(counts, weights[missing].sum())[0]


def gain_ratio(y, branches, sample_weight=None):
    """Information gain dibagi split information; split konstan bernilai nol."""
    y, weights = _labels_weights(y, sample_weight)
    branches = np.asarray(branches)
    if branches.shape != y.shape:
        raise ValueError("branches harus satu dimensi dengan panjang sama dengan y.")
    missing = _missing(branches)
    values, inverse = np.unique(branches[~missing], return_inverse=True)
    counts = np.zeros((len(values), 2))
    np.add.at(counts, (inverse, y[~missing]), weights[~missing])
    gain, split_info = _partition_scores(counts, weights[missing].sum())
    return gain / split_info if split_info > 1e-12 else 0.0


@dataclass
class TreeNode:
    """Statistik kelas berbobot dan aturan split; children kosong berarti daun."""

    class_counts: np.ndarray
    n_samples: int
    depth: int
    feature: int | None = None
    threshold: float | None = None
    categorical: bool = False
    children: dict = field(default_factory=dict)
    branch_probabilities: dict = field(default_factory=dict)
    information_gain: float = 0.0
    gain_ratio: float = 0.0

    def is_leaf(self):
        """Periksa apakah simpul tidak memiliki anak."""
        return not self.children

    @property
    def probability(self):
        """Hitung peluang kelas 1 dari jumlah kelas berbobot."""
        return float(self.class_counts[1] / self.class_counts.sum())


@dataclass
class _Split:
    feature: int
    threshold: float | None
    categories: tuple | None
    gain: float
    ratio: float
    probabilities: np.ndarray


class DecisionTree:
    """Pohon berbasis C4.5 mengacu pada salindia kuliah Decision Tree.pdf dan DTL.pdf."""

    def __init__(self, categorical_features=(), max_depth=20,
                 min_samples_split=2, min_samples_leaf=1.0, min_gain=1e-12):
        """Simpan indeks fitur kategorikal dan parameter penghentian pertumbuhan pohon."""
        self.categorical_features = tuple(categorical_features)
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.min_gain = min_gain

    def _validate_parameters(self, n_features):
        """Validasi parameter pertumbuhan dan indeks fitur kategorikal."""
        if (self.max_depth is not None and
                (isinstance(self.max_depth, bool)
                 or not isinstance(self.max_depth, Integral) or self.max_depth < 0)):
            raise ValueError("max_depth harus integer nonnegatif atau None.")
        if (isinstance(self.min_samples_split, bool)
                or not isinstance(self.min_samples_split, Integral)
                or self.min_samples_split < 2):
            raise ValueError("min_samples_split harus integer >= 2.")
        for name, value, positive in [("min_samples_leaf", self.min_samples_leaf, True),
                                      ("min_gain", self.min_gain, False)]:
            if (isinstance(value, bool) or not isinstance(value, Real)
                    or not np.isfinite(value) or value < 0 or (positive and value == 0)):
                raise ValueError(f"{name} harus finite dan {'positif' if positive else 'nonnegatif'}.")
        features = self.categorical_features
        if any(isinstance(f, bool) or not isinstance(f, Integral)
               or f < 0 or f >= n_features for f in features):
            raise ValueError("Indeks fitur kategorikal di luar rentang atau bukan integer.")
        if len(set(features)) != len(features):
            raise ValueError("Indeks fitur kategorikal tidak boleh duplikat.")

    @staticmethod
    def _validate_X(X, categorical_features, n_features=None):
        """Validasi dan salin fitur numerik atau kategorikal serta seragamkan nilai missing."""
        X = np.asarray(X)
        if X.ndim != 2 or X.shape[1] == 0:
            raise ValueError("X harus array dua dimensi dengan setidaknya satu fitur.")
        if np.iscomplexobj(X):
            raise ValueError("X tidak boleh berisi bilangan kompleks.")
        if n_features is not None and X.shape[1] != n_features:
            raise ValueError("Jumlah fitur berbeda dengan data training.")
        result = np.array(X, dtype=object if categorical_features else float, copy=True)
        for feature in range(X.shape[1]):
            column = result[:, feature]
            missing = _missing(column)
            known = column[~missing]
            if feature in categorical_features:
                strings = all(isinstance(v, (str, np.str_)) for v in known)
                numbers = all(isinstance(v, Real) and np.isfinite(v) for v in known)
                if not (strings or numbers):
                    raise ValueError("Kategori per kolom harus seragam: string atau angka finite.")
            else:
                try:
                    values = np.asarray(known, dtype=float)
                except (TypeError, ValueError) as exc:
                    raise ValueError("Kolom nonkategorikal harus numerik.") from exc
                if not np.isfinite(values).all():
                    raise ValueError("Fitur numerik tidak boleh berisi infinity.")
                column[~missing] = values
            column[missing] = np.nan
        return result

    def _best_numeric(self, x, y, weights, feature):
        """Cari ambang numerik dengan gain ratio terbaik menggunakan hitungan kelas kumulatif."""
        x = np.asarray(x, dtype=float)
        known = ~np.isnan(x)
        order = np.argsort(x[known], kind="stable")
        values, labels, w = x[known][order], y[known][order], weights[known][order]
        if len(values) < 2:
            return None
        positions = np.flatnonzero(values[:-1] < values[1:])
        if not len(positions):
            return None
        cumulative = np.cumsum(w)
        positive = np.cumsum(w * labels)
        left_w, left_pos = cumulative[positions], positive[positions]
        right_w, right_pos = cumulative[-1] - left_w, positive[-1] - left_pos
        known_w, missing_w = cumulative[-1], weights[~known].sum()
        total = known_w + missing_w
        masses = np.column_stack((left_w, right_w))
        probabilities = masses / known_w
        valid = np.all(probabilities * total >= self.min_samples_leaf - 1e-12, axis=1)
        left = np.column_stack((left_w - left_pos, left_pos))
        right = np.column_stack((right_w - right_pos, right_pos))
        parent_h = _entropy_counts([known_w - positive[-1], positive[-1]])
        gain = np.maximum(0.0, known_w / total *
                          (parent_h - (left_w * _entropy_counts(left)
                                       + right_w * _entropy_counts(right)) / known_w))
        split_counts = np.column_stack((masses, np.full(len(masses), missing_w)))
        split_info = _entropy_counts(split_counts)
        ratios = np.divide(gain, split_info, out=np.zeros_like(gain), where=split_info > 1e-12)
        valid &= (gain > self.min_gain) & (split_info > 1e-12)
        if not valid.any():
            return None
        best = int(np.argmax(np.where(valid, ratios, -np.inf)))
        a, b = values[positions[best]], values[positions[best] + 1]
        threshold = a / 2.0 + b / 2.0
        if not a <= threshold < b:
            threshold = a
        return _Split(feature, float(threshold), None, float(gain[best]),
                      float(ratios[best]), probabilities[best])

    def _best_categorical(self, x, y, weights, feature):
        """Evaluasi split per kategori berdasarkan gain ratio dan batas bobot daun."""
        known = ~_missing(x)
        categories, inverse = np.unique(x[known], return_inverse=True)
        if len(categories) < 2:
            return None
        counts = np.zeros((len(categories), 2))
        np.add.at(counts, (inverse, y[known]), weights[known])
        masses = counts.sum(axis=1)
        probabilities = masses / masses.sum()
        if (probabilities * weights.sum() < self.min_samples_leaf - 1e-12).any():
            return None
        gain, split_info = _partition_scores(counts, weights[~known].sum())
        if gain <= self.min_gain or split_info <= 1e-12:
            return None
        return _Split(feature, None, tuple(categories), gain, gain / split_info, probabilities)

    @staticmethod
    def _branches(x, split):
        """Tentukan indeks cabang setiap sampel dengan penanda -1 untuk nilai missing."""
        branches = np.full(len(x), -1, dtype=int)
        known = ~_missing(x)
        if split.categories is None:
            branches[known] = (np.asarray(x[known], dtype=float) > split.threshold).astype(int)
        else:
            for branch, category in enumerate(split.categories):
                branches[known & (x == category)] = branch
        return branches

    def fit(self, X, y):
        """Latih pohon dari data training dengan bobot pecahan tanpa memodifikasi input."""
        raw = np.asarray(X)
        if raw.ndim != 2:
            raise ValueError("X harus array dua dimensi.")
        self._validate_parameters(raw.shape[1])
        categorical = frozenset(self.categorical_features)
        X = self._validate_X(X, categorical)
        y, weights = _labels_weights(y)
        if len(y) == 0 or len(y) != len(X):
            raise ValueError("X dan y harus tidak kosong dan memiliki jumlah baris sama.")
        root = TreeNode(np.bincount(y, minlength=2).astype(float), len(y), 0)
        n_nodes, depth = 1, 0
        stack = [(root, np.arange(len(y)), weights, categorical)]
        while stack:
            node, indices, w, available_categories = stack.pop()
            if (np.count_nonzero(node.class_counts) < 2
                    or len(indices) < self.min_samples_split
                    or w.sum() < 2 * self.min_samples_leaf
                    or (self.max_depth is not None and node.depth >= self.max_depth)):
                continue
            candidates = []
            for feature in range(X.shape[1]):
                if feature in categorical and feature not in available_categories:
                    continue
                search = self._best_categorical if feature in categorical else self._best_numeric
                candidate = search(X[indices, feature], y[indices], w, feature)
                if candidate is not None:
                    candidates.append(candidate)
            if not candidates:
                continue
            average_gain = np.mean([s.gain for s in candidates])
            eligible = [s for s in candidates if s.gain >= average_gain - 1e-12]
            split = max(eligible, key=lambda s: s.ratio)
            branches = self._branches(X[indices, split.feature], split)
            missing = branches == -1
            node.feature, node.threshold = split.feature, split.threshold
            node.categorical = split.categories is not None
            node.information_gain, node.gain_ratio = split.gain, split.ratio
            keys = split.categories if node.categorical else (0, 1)
            next_categories = available_categories - {split.feature} if node.categorical else available_categories
            for branch, key in enumerate(keys):
                probability = float(split.probabilities[branch])
                mask = (branches == branch) | missing
                child_indices = indices[mask]
                child_w = w[mask] * np.where(missing[mask], probability, 1.0)
                counts = np.bincount(y[child_indices], weights=child_w, minlength=2)
                child = TreeNode(counts, len(child_indices), node.depth + 1)
                node.children[key] = child
                node.branch_probabilities[key] = probability
                n_nodes += 1
                depth = max(depth, child.depth)
                stack.append((child, child_indices, child_w, next_categories))
        self.root = root
        self.n_features_in_ = X.shape[1]
        self.categorical_features_ = categorical
        self.n_nodes_, self.depth_ = n_nodes, depth
        self.classes_ = np.array([0, 1])
        return self

    def _check_fitted(self):
        """Pastikan model sudah dilatih sebelum digunakan."""
        if not hasattr(self, "root"):
            raise RuntimeError("Model belum dilatih; panggil fit terlebih dahulu.")

    def predict_proba(self, X):
        """Peluang kelas 1; missing/kategori baru menggabungkan semua cabang."""
        self._check_fitted()
        X = self._validate_X(X, self.categorical_features_, self.n_features_in_)
        result = np.zeros(len(X))
        stack = [(self.root, np.arange(len(X)), np.ones(len(X)))]
        while stack:
            node, indices, w = stack.pop()
            if not len(indices):
                continue
            if node.is_leaf():
                result[indices] += w * node.probability
                continue
            x = X[indices, node.feature]
            known = ~_missing(x)
            if node.categorical:
                matched = np.zeros(len(x), dtype=bool)
                masks = {}
                for key in node.children:
                    masks[key] = known & (x == key)
                    matched |= masks[key]
            else:
                left = np.zeros(len(x), dtype=bool)
                left[known] = np.asarray(x[known], dtype=float) <= node.threshold
                masks = {0: known & left, 1: known & ~left}
                matched = known
            unknown = ~matched
            for key, child in node.children.items():
                mask = masks[key] | unknown
                child_w = w[mask] * np.where(unknown[mask], node.branch_probabilities[key], 1.0)
                stack.append((child, indices[mask], child_w))
        return np.clip(result, 0.0, 1.0)

    def predict(self, X, threshold=0.5):
        """Kelas 1 jika probabilitas >= threshold, termasuk saat tepat sama."""
        if (not isinstance(threshold, Real) or not np.isfinite(threshold)
                or not 0 <= threshold <= 1):
            raise ValueError("threshold harus angka finite dalam [0, 1].")
        return (self.predict_proba(X) >= threshold).astype(np.int64)

    def plot_tree(self, path, top_n=3, *, feature_names=None):
        """Simpan gambar yang membuka paling banyak top_n simpul pemisah dari akar per tingkat."""
        self._check_fitted()
        if isinstance(top_n, bool) or not isinstance(top_n, Integral) or top_n < 1:
            raise ValueError("top_n harus bilangan bulat positif.")
        names = ([f"X[{i}]" for i in range(self.n_features_in_)]
                 if feature_names is None else list(feature_names))
        if len(names) != self.n_features_in_:
            raise ValueError("Jumlah nama fitur harus sama dengan jumlah kolom latih.")

        import matplotlib.pyplot as plt

        nodes, edges = [self.root], {}
        for index, node in enumerate(nodes):
            if node.is_leaf():
                continue
            if len(edges) == top_n:
                break
            children = []
            for key, child in node.children.items():
                children.append((len(nodes), key))
                nodes.append(child)
            edges[index] = children

        terminals = [i for i in range(len(nodes)) if i not in edges]
        # Susun terminal per subpohon agar garis antarcabang tidak berpotongan.
        row, position = 0, {}
        stack = [0]
        while stack:
            index = stack.pop()
            if index in edges:
                stack.extend(child for child, _ in reversed(edges[index]))
            else:
                position[index] = row
                row += 1
        for index in reversed(edges):
            children = edges[index]
            position[index] = (position[children[0][0]] + position[children[-1][0]]) / 2

        depth = max(node.depth for node in nodes)
        fig, ax = plt.subplots(figsize=(max(5, 4.4 * depth + 3), max(3.3, 0.68 * len(terminals) + 1)))
        try:
            for index, children in edges.items():
                node = nodes[index]
                x, y = node.depth, position[index]
                for child, key in children:
                    child_y = position[child]
                    ax.plot([x, x + 0.5, x + 0.5, x + 1],
                            [y, y, child_y, child_y], color="#64748b", linewidth=1)
                    label = str(key) if node.categorical else ("≤" if key == 0 else ">")
                    ax.text(x + 0.63, child_y, label, ha="center", va="center",
                            fontsize=8, parse_math=False,
                            bbox={"facecolor": "white", "edgecolor": "none", "pad": 1})

            for index, node in enumerate(nodes):
                if index in edges:
                    name = str(names[node.feature])
                    rule = name if node.categorical else f"{name}\nAmbang: {node.threshold:.6g}"
                    label, color = f"Pemisah: {rule}", "#dbeafe"
                elif node.is_leaf():
                    label, color = "Daun", "#dcfce7"
                else:
                    label, color = "Cabang diringkas", "#fef3c7"
                label += f"\nP(batal): {node.probability:.1%} · bobot: {node.class_counts.sum():.1f}"
                ax.text(node.depth, position[index], label, ha="center", va="center", fontsize=8,
                        parse_math=False, bbox={"boxstyle": "round,pad=0.35", "facecolor": color,
                                               "edgecolor": "#64748b"})

            ax.set_xlim(-0.35, depth + 0.35)
            ax.set_ylim(len(terminals) - 0.35, -0.65)
            ax.set_axis_off()
            ax.set_title(f"Pohon keputusan C4.5 · {len(edges)} simpul pemisah ditampilkan (top_n={top_n})")
            fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
        finally:
            plt.close(fig)

    def save(self, path):
        """Simpan model terlatih dengan pickle dari standard library."""
        self._check_fitted()
        with open(path, "wb") as file:
            pickle.dump(self, file, protocol=pickle.HIGHEST_PROTOCOL)

    @classmethod
    def load(cls, path):
        """Muat hanya berkas pickle tepercaya; pickle dapat mengeksekusi kode."""
        with open(path, "rb") as file:
            model = pickle.load(file)
        if not isinstance(model, cls):
            raise ValueError("Berkas tidak berisi model DecisionTree.")
        model._check_fitted()
        return model

In [ ]:
dtl_model = DecisionTree(categorical_features=tree_categorical).fit(T_train, y_train)
dtl_val_proba = dtl_model.predict_proba(T_val)

print(f'Nodes: {dtl_model.n_nodes_}  Depth: {dtl_model.depth_}')
pd.Series(scores(y_val, dtl_val_proba), name='validation')

### Decision Tree Visualization (Bonus)

Gambar menampilkan pohon C4.5 yang telah dilatih pada data latih. Parameter `top_n` membatasi jumlah simpul pemisah yang dibuka dari akar per tingkat. Simpul yang masih memiliki keturunan setelah batas tercapai diberi label "Cabang diringkas". Bobot menunjukkan kontribusi data latih pada simpul, termasuk bobot pecahan dari nilai hilang.

Tampilan dengan `top_n=3` merangkum aturan awal. Tampilan dengan `top_n=7` menunjukkan aturan yang lebih rinci.

In [ ]:
for dtl_top_n in (3, 7):
    dtl_plot_path = CFG.OUTPUT_DIR / f'decision_tree_top_{dtl_top_n}.png'
    dtl_model.plot_tree(dtl_plot_path, top_n=dtl_top_n, feature_names=T_train.columns)
    display(Image(filename=str(dtl_plot_path)))

## Logistic Regression

Eksperimen menggunakan regularisasi L1, L2, dan Elastic Net. Tabel menampilkan metrik, jumlah iterasi, status konvergensi, dan jumlah koefisien nol. Elastic Net terpilih berdasarkan akurasi validasi dengan akurasi 0,7980 dan F1 kelas batal 0,6550.

In [ ]:
"""Logistic Regression from scratch (NumPy) dengan regularisasi L1, L2, dan Elastic Net.

Fungsi tujuan (rata-rata log-loss ditambah penalti):

    J(w, b) = (1/n) sum_i [ log(1 + exp(z_i)) - y_i z_i ] + R(w),   z_i = x_i . w + b

    L1          : R(w) = alpha * ||w||_1
    L2          : R(w) = alpha/2 * ||w||_2^2
    Elastic Net : R(w) = alpha * ( l1_ratio * ||w||_1 + (1 - l1_ratio)/2 * ||w||_2^2 )

Bias `b` tidak diregularisasi. Optimasi: gradient descent (batch penuh atau mini-batch). Bagian penalti L1
tidak mulus di 0, jadi dipakai langkah proximal (soft-threshold) setelah langkah gradien. Dengan cara ini
bobot yang tidak berguna bernilai tepat 0, bukan hanya mendekati 0.

Hubungan dengan scikit-learn: objektif sklearn memakai C * sum(loss) + penalti, sehingga alpha = 1 / (C * n).

Referensi:
- M. L. Khodra, "Logistic Regression: What & Why" dan "Stochastic Gradient Ascent", YouTube, 2021 (rujukan kuliah).
- A. Beck & M. Teboulle (2009). A Fast Iterative Shrinkage-Thresholding Algorithm for Linear Inverse Problems.
  SIAM Journal on Imaging Sciences 2(1). (langkah proximal / soft-threshold untuk L1)
- H. Zou & T. Hastie (2005). Regularization and Variable Selection via the Elastic Net.
  Journal of the Royal Statistical Society B 67(2). (bentuk penalti Elastic Net)
"""


def sigmoid(z):
    """Sigmoid stabil secara numerik: tidak overflow untuk |z| besar."""
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    e = np.exp(z[~pos])
    out[~pos] = e / (1.0 + e)
    return out


def log_loss(y, p, eps=1e-15):
    """Rata-rata binary cross-entropy. `p` dipotong ke [eps, 1-eps] agar log(0) aman."""
    y = np.asarray(y, dtype=float)
    p = np.clip(np.asarray(p, dtype=float), eps, 1 - eps)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


class Regularizer:
    """Penalti R(w). `grad_smooth` adalah gradien bagian mulus (L2), `prox` langkah proximal bagian L1."""

    def penalty(self, w):
        raise NotImplementedError

    def grad_smooth(self, w):
        raise NotImplementedError

    def prox(self, w, step):
        raise NotImplementedError


def _soft_threshold(w, t):
    return np.sign(w) * np.maximum(np.abs(w) - t, 0.0)


class NoPenalty(Regularizer):
    def penalty(self, w):
        return 0.0

    def grad_smooth(self, w):
        return np.zeros_like(w)

    def prox(self, w, step):
        return w


class L1(Regularizer):
    def __init__(self, alpha):
        self.alpha = alpha

    def penalty(self, w):
        return self.alpha * float(np.abs(w).sum())

    def grad_smooth(self, w):
        return np.zeros_like(w)

    def prox(self, w, step):
        return _soft_threshold(w, step * self.alpha)


class L2(Regularizer):
    def __init__(self, alpha):
        self.alpha = alpha

    def penalty(self, w):
        return 0.5 * self.alpha * float(w @ w)

    def grad_smooth(self, w):
        return self.alpha * w

    def prox(self, w, step):
        return w


class ElasticNet(Regularizer):
    def __init__(self, alpha, l1_ratio=0.5):
        self.alpha, self.l1_ratio = alpha, l1_ratio

    def penalty(self, w):
        return self.alpha * (self.l1_ratio * float(np.abs(w).sum())
                             + 0.5 * (1 - self.l1_ratio) * float(w @ w))

    def grad_smooth(self, w):
        return self.alpha * (1 - self.l1_ratio) * w

    def prox(self, w, step):
        return _soft_threshold(w, step * self.alpha * self.l1_ratio)


def make_regularizer(penalty, alpha, l1_ratio=0.5):
    if penalty == "none":
        return NoPenalty()
    if penalty == "l1":
        return L1(alpha)
    if penalty == "l2":
        return L2(alpha)
    if penalty == "elasticnet":
        return ElasticNet(alpha, l1_ratio)
    raise ValueError("penalty harus 'none', 'l1', 'l2', atau 'elasticnet'.")


class LogisticRegression:
    """Klasifikasi biner. Fitur harus numerik, tanpa NaN (pakai tampilan numerik dari preprocessing).

    lr          : laju belajar (fitur terstandarisasi: 0.1 sampai 0.5 biasanya aman).
    n_iter      : jumlah epoch maksimum.
    penalty     : "none", "l1", "l2", atau "elasticnet".
    alpha       : kekuatan regularisasi. l1_ratio: porsi L1 pada elasticnet (0 = L2 murni, 1 = L1 murni).
    tol         : berhenti jika perubahan objektif antar epoch < tol.
    batch_size  : None = batch penuh, bilangan bulat = mini-batch (diacak per epoch dengan random_state).
    record_every: riwayat parameter dicatat tiap sekian epoch (epoch 0 dan terakhir selalu dicatat).

    Setelah fit: coef_ (d,), intercept_, n_iter_, converged_, history_ = {"epoch", "loss", "params"}
    dengan params[:, 0] = bias dan params[:, 1:] = bobot; loss = objektif (log-loss + penalti).
    """

    def __init__(self, lr=0.1, n_iter=2000, penalty="l2", alpha=0.01, l1_ratio=0.5,
                 tol=1e-8, batch_size=None, random_state=42, record_every=1):
        self.lr, self.n_iter, self.penalty, self.alpha = lr, n_iter, penalty, alpha
        self.l1_ratio, self.tol, self.batch_size = l1_ratio, tol, batch_size
        self.random_state, self.record_every = random_state, record_every

    def _validate_params(self):
        make_regularizer(self.penalty, self.alpha, self.l1_ratio)
        if not self.lr > 0:
            raise ValueError("lr harus > 0.")
        if int(self.n_iter) < 1:
            raise ValueError("n_iter harus >= 1.")
        if self.alpha < 0:
            raise ValueError("alpha harus >= 0.")
        if not 0 <= self.l1_ratio <= 1:
            raise ValueError("l1_ratio harus di [0, 1].")
        if self.batch_size is not None and int(self.batch_size) < 1:
            raise ValueError("batch_size harus None atau >= 1.")
        if int(self.record_every) < 1:
            raise ValueError("record_every harus >= 1.")

    @staticmethod
    def _check_X(X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError("X harus 2 dimensi (n_baris, n_fitur).")
        if not np.isfinite(X).all():
            raise ValueError("X mengandung NaN atau inf; imputasi dulu di preprocessing.")
        return X

    def fit(self, X, y):
        self._validate_params()
        X = self._check_X(X)
        y = np.asarray(y).astype(float)
        if y.shape != (len(X),):
            raise ValueError("y harus 1 dimensi dengan panjang sama dengan X.")
        if not np.isin(y, (0.0, 1.0)).all():
            raise ValueError("y harus berisi 0 atau 1.")

        n, d = X.shape
        reg = make_regularizer(self.penalty, self.alpha, self.l1_ratio)
        rng = np.random.default_rng(self.random_state)
        w, b = np.zeros(d), 0.0

        def objective():
            return log_loss(y, sigmoid(X @ w + b)) + reg.penalty(w)

        prev = objective()
        epochs, losses, params = [0], [prev], [np.r_[b, w]]
        self.converged_ = False
        for epoch in range(1, int(self.n_iter) + 1):
            if self.batch_size is None:
                batches = [np.arange(n)]
            else:
                order = rng.permutation(n)
                batches = np.array_split(order, max(1, -(-n // int(self.batch_size))))
            for idx in batches:
                err = sigmoid(X[idx] @ w + b) - y[idx]
                grad_w = X[idx].T @ err / len(idx) + reg.grad_smooth(w)
                w = reg.prox(w - self.lr * grad_w, self.lr)
                b -= self.lr * err.mean()
            obj = objective()
            if not np.isfinite(obj):
                raise ValueError("Loss tidak finite: lr terlalu besar atau fitur belum diskalakan.")
            if epoch % int(self.record_every) == 0:
                epochs.append(epoch)
                losses.append(obj)
                params.append(np.r_[b, w])
            if abs(prev - obj) < self.tol:
                self.converged_ = True
                break
            prev = obj
        if epochs[-1] != epoch:
            epochs.append(epoch)
            losses.append(obj)
            params.append(np.r_[b, w])

        self.coef_, self.intercept_, self.n_iter_ = w, float(b), epoch
        self.n_features_ = d
        self.history_ = {"epoch": np.array(epochs), "loss": np.array(losses), "params": np.vstack(params)}
        return self

    def _check_fitted(self):
        if not hasattr(self, "coef_"):
            raise RuntimeError("Model belum di-fit.")

    def decision_function(self, X):
        self._check_fitted()
        X = self._check_X(X)
        if X.shape[1] != self.n_features_:
            raise ValueError(f"X punya {X.shape[1]} fitur, model dilatih dengan {self.n_features_}.")
        return X @ self.coef_ + self.intercept_

    def predict_proba(self, X):
        """Peluang kelas 1, array (m,)."""
        return sigmoid(self.decision_function(X))

    def predict(self, X, threshold=0.5):
        """Kelas 1 jika peluang >= threshold."""
        if not np.isfinite(threshold) or not 0 <= threshold <= 1:
            raise ValueError("threshold harus angka finite dalam [0, 1].")
        return (self.predict_proba(X) >= threshold).astype(int)

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            model = pickle.load(f)
        if not isinstance(model, cls):
            raise TypeError(f"Berkas {path} berisi {type(model).__name__}, bukan {cls.__name__}.")
        return model

    def plot_loss_contour_video(self, X, y, path, fps=20, max_frames=150, max_points=2000,
                                feature_name="x", grid=80):
        """[BONUS 3] Video kontur log-loss pada bidang (theta0, theta1) beserta lintasan parameter saat training.

        Model harus dilatih dengan tepat 1 fitur (theta0 = bias, theta1 = bobot). X, y: data yang sama dengan
        saat fit (disubsampel sampai max_points baris hanya untuk menggambar kontur). Format mp4 bila ffmpeg
        tersedia; jika tidak, GIF. Mengembalikan path berkas yang benar-benar ditulis.
        """
        self._check_fitted()
        if self.n_features_ != 1:
            raise ValueError("Video butuh model dengan tepat 1 fitur (2 parameter: theta0 dan theta1).")
        from matplotlib import animation
        from matplotlib.figure import Figure

        X, y = self._check_X(X), np.asarray(y).astype(float)
        if len(X) > max_points:
            keep = np.random.default_rng(self.random_state).choice(len(X), max_points, replace=False)
            X, y = X[keep], y[keep]
        x = X[:, 0]

        traj = self.history_["params"]
        loss_hist = self.history_["loss"]
        frames = np.unique(np.linspace(0, len(traj) - 1, min(max_frames, len(traj))).round().astype(int))

        lo, hi = traj.min(axis=0), traj.max(axis=0)
        pad = np.maximum(0.3 * (hi - lo), 0.5)
        t0 = np.linspace(lo[0] - pad[0], hi[0] + pad[0], grid)
        t1 = np.linspace(lo[1] - pad[1], hi[1] + pad[1], grid)
        T0, T1 = np.meshgrid(t0, t1)
        Z = np.empty_like(T0)
        for r in range(grid):
            z = T0[r][:, None] + T1[r][:, None] * x[None, :]
            Z[r] = (np.logaddexp(0, z) - y[None, :] * z).mean(axis=1)

        fig = Figure(figsize=(7, 5.5))
        ax = fig.subplots()
        cs = ax.contour(T0, T1, Z, levels=18, cmap="viridis")
        ax.clabel(cs, inline=True, fontsize=7, fmt="%.2f")
        ax.set_xlabel(r"$\theta_0$ (bias)")
        ax.set_ylabel(rf"$\theta_1$ (bobot {feature_name})")
        (line,) = ax.plot([], [], "-", color="#c0392b", lw=1.8)
        (dot,) = ax.plot([], [], "o", color="#c0392b", ms=6)
        ax.plot(traj[0, 0], traj[0, 1], "ks", ms=6, label="awal")
        ax.legend(loc="upper right")
        title = ax.set_title("")

        def draw(k):
            i = frames[k]
            line.set_data(traj[: i + 1, 0], traj[: i + 1, 1])
            dot.set_data([traj[i, 0]], [traj[i, 1]])
            title.set_text(f"epoch {int(self.history_['epoch'][i])}, objektif {loss_hist[i]:.4f}")
            return line, dot, title

        anim = animation.FuncAnimation(fig, draw, frames=len(frames), blit=False)
        path = str(path)
        if path.lower().endswith(".mp4") and shutil.which("ffmpeg"):
            writer = animation.FFMpegWriter(fps=fps)
        else:
            if path.lower().endswith(".mp4"):
                path = path[:-4] + ".gif"
            writer = animation.PillowWriter(fps=fps)
        anim.save(path, writer=writer, dpi=110)
        return path

In [ ]:
lr_models, lr_val_proba, lr_table = {}, {}, []
for penalty in ['l1', 'l2', 'elasticnet']:
    model = LogisticRegression(penalty=penalty).fit(X_train, y_train)
    lr_models[penalty] = model
    lr_val_proba[penalty] = model.predict_proba(X_val)
    lr_table.append({'penalty': penalty, 'epochs': model.n_iter_, 'converged': model.converged_,
                     'zero weights': int((model.coef_ == 0).sum()), **scores(y_val, lr_val_proba[penalty])})
pd.DataFrame(lr_table).set_index('penalty').round(4)

## Support Vector Machine

<!-- rubric v3: total slack, slack_total -->

Eksperimen menguji kernel linear, polinomial, dan Gaussian pada subsampel 4.000 baris. Satu eksperimen tambahan menggunakan kernel Gaussian pada subsampel 12.000 baris.

In [ ]:
"""SVM from scratch (NumPy) dengan SMO.

Referensi:
- Platt, J. (1998). Sequential Minimal Optimization: A Fast Algorithm for Training Support Vector Machines. MSR-TR-98-14.
- Keerthi, S. S., dkk. (2001). Improvements to Platt's SMO Algorithm for SVM Classifier Design. Neural Computation 13(3).
  (pemilihan pasangan "maximal violating pair" yang dipakai di sini)
- Platt, J. (1999). Probabilistic Outputs for Support Vector Machines. (Platt scaling)
- Lin, H.-T., Lin, C.-J., Weng, R. C. (2007). A Note on Platt's Probabilistic Outputs for Support Vector Machines.
  Machine Learning 68(3). (target ter-smoothing untuk Platt scaling)
"""


class LinearKernel:
    def compute(self, A, B):
        return A @ B.T


class PolynomialKernel:
    def __init__(self, degree=3, gamma=None, coef0=1.0):
        self.degree, self.gamma, self.coef0 = degree, gamma, coef0

    def compute(self, A, B):
        gamma = self.gamma if self.gamma is not None else 1.0 / A.shape[1]
        return (gamma * (A @ B.T) + self.coef0) ** self.degree


class GaussianKernel:
    def __init__(self, gamma=None):
        self.gamma = gamma

    def compute(self, A, B):
        gamma = self.gamma if self.gamma is not None else 1.0 / A.shape[1]
        sq = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * (A @ B.T)
        return np.exp(-gamma * np.maximum(sq, 0))


def _stratified_subsample(y, size, rng):
    """Indeks subsampel berstratifikasi sebanyak `size`, sisanya dikembalikan terpisah."""
    picked = []
    for c in np.unique(y):
        idx = np.flatnonzero(y == c)
        picked.append(rng.choice(idx, round(size * len(idx) / len(y)), replace=False))
    picked = np.sort(np.concatenate(picked))
    return picked, np.setdiff1d(np.arange(len(y)), picked)


def _fit_platt(f, y, n_iter=100):
    """Cari A, B pada p = 1 / (1 + exp(A f + B)) dengan Newton (Lin dkk., 2007)."""
    n_pos, n_neg = (y == 1).sum(), (y == 0).sum()
    t = np.where(y == 1, (n_pos + 1) / (n_pos + 2), 1 / (n_neg + 2))
    A, B = 0.0, np.log((n_neg + 1) / (n_pos + 1))

    def loss(A, B):
        z = A * f + B  # log-loss stabil: t*z + log(1 + exp(-z))
        return np.sum(t * z + np.logaddexp(0, -z))

    for _ in range(n_iter):
        p = 1 / (1 + np.exp(np.clip(A * f + B, -500, 500)))
        g = np.array([np.dot(f, t - p), np.sum(t - p)])
        w = p * (1 - p)
        H = np.array([[np.dot(f * f, w), np.dot(f, w)], [np.dot(f, w), w.sum()]]) + 1e-12 * np.eye(2)
        step = np.linalg.solve(H, g)
        old, s = loss(A, B), 1.0
        while s > 1e-8 and loss(A - s * step[0], B - s * step[1]) > old:
            s /= 2
        A, B = A - s * step[0], B - s * step[1]
        if np.abs(g).max() < 1e-5:
            break
    return A, B


class SVM:
    """SVM klasifikasi biner, dual diselesaikan dengan SMO.

    Strategi skala data: matriks kernel O(n^2) untuk 27 ribu baris tidak muat, jadi bila n > max_samples,
    SVM dilatih pada subsampel berstratifikasi sebanyak max_samples (matriks kernel penuh di-cache).
    Baris sisanya dipakai untuk Platt scaling, sehingga kalibrasi tidak dilakukan pada data latih SVM.
    """

    def __init__(self, kernel=None, C=1.0, tol=1e-3, max_iter=100_000, max_samples=4000,
                 n_calib=5000, random_state=42):
        self.kernel = kernel if kernel is not None else GaussianKernel()
        self.C, self.tol, self.max_iter = C, tol, max_iter
        self.max_samples, self.n_calib, self.random_state = max_samples, n_calib, random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y01 = np.asarray(y).astype(int)
        rng = np.random.default_rng(self.random_state)
        if len(y01) > self.max_samples:
            tr, rest = _stratified_subsample(y01, self.max_samples, rng)
        else:
            tr, rest = np.arange(len(y01)), np.array([], dtype=int)

        Xs, ys = X[tr], np.where(y01[tr] == 1, 1.0, -1.0)
        alpha = self._smo(self.kernel.compute(Xs, Xs), ys)

        sv = alpha > 1e-8
        self.support_vectors_ = Xs[sv]
        self.dual_coef_ = alpha[sv] * ys[sv]  # alpha_i * y_i
        self.support_ = tr[sv]

        if len(rest):  # kalibrasi pada baris yang tidak dipakai melatih SVM
            cal = rest if len(rest) <= self.n_calib else rng.choice(rest, self.n_calib, replace=False)
        else:  # shortcut: data kecil, kalibrasi pada data latih (bias optimis); pakai CV bila perlu
            cal = tr
        self.platt_ = _fit_platt(self.decision_function(X[cal]), y01[cal])
        return self

    def _smo(self, K, y):
        """Minimalkan 1/2 a^T Q a - e^T a, 0 <= a <= C, y^T a = 0, dengan Q_ij = y_i y_j K_ij."""
        n, C = len(y), self.C
        alpha = np.zeros(n)
        G = -np.ones(n)  # gradien Q a - e
        diagK = np.diag(K).copy()
        self.converged_ = False
        for it in range(self.max_iter):
            up = ((y > 0) & (alpha < C)) | ((y < 0) & (alpha > 0))
            low = ((y > 0) & (alpha > 0)) | ((y < 0) & (alpha < C))
            score = -y * G
            i = np.flatnonzero(up)[np.argmax(score[up])]
            j = np.flatnonzero(low)[np.argmin(score[low])]
            gap = score[i] - score[j]
            if gap < self.tol:  # KKT terpenuhi dalam toleransi
                self.converged_ = True
                break
            # Langkah t sepanjang a_i += y_i t, a_j -= y_j t (menjaga y^T a), lalu dipotong ke kotak [0, C].
            eta = max(diagK[i] + diagK[j] - 2 * K[i, j], 1e-12)
            t = min(gap / eta,
                    C - alpha[i] if y[i] > 0 else alpha[i],
                    alpha[j] if y[j] > 0 else C - alpha[j])
            alpha[i] += y[i] * t
            alpha[j] -= y[j] * t
            G += y * t * (K[:, i] - K[:, j])
        self.n_iter_ = it + 1
        self.kkt_gap_ = gap

        # b dari support vector bebas (0 < a < C): y_t f(x_t) = 1  =>  b = -y_t G_t
        free = (alpha > 1e-8) & (alpha < C - 1e-8)
        self.b_ = (-y * G)[free].mean() if free.any() else (score[up].max() + score[low].min()) / 2
        return alpha

    def decision_function(self, X):
        X = np.asarray(X, dtype=float)
        return self.kernel.compute(X, self.support_vectors_) @ self.dual_coef_ + self.b_

    def predict_proba(self, X):
        A, B = self.platt_
        return 1 / (1 + np.exp(np.clip(A * self.decision_function(X) + B, -500, 500)))

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def plot_decision_boundary(self, X, y, path, feature_names=("x1", "x2"), title=None, max_points=1500):
        """[BONUS 4] Bidang pemisah f(x)=0, margin f(x)=±1, dan support vector. Model harus dilatih pada 2 fitur."""
        X, y = np.asarray(X, dtype=float), np.asarray(y).astype(int)
        if X.shape[1] != 2 or self.support_vectors_.shape[1] != 2:
            raise ValueError("plot_decision_boundary butuh model dan X dengan tepat 2 fitur")
        lo, hi = np.percentile(X, [1, 99], axis=0)  # potong outlier agar plot terbaca
        pad = 0.1 * (hi - lo)
        g1, g2 = np.meshgrid(np.linspace(lo[0] - pad[0], hi[0] + pad[0], 300),
                             np.linspace(lo[1] - pad[1], hi[1] + pad[1], 300))
        f = self.decision_function(np.c_[g1.ravel(), g2.ravel()]).reshape(g1.shape)

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.contourf(g1, g2, f, levels=[-np.inf, 0, np.inf], colors=["#cfe3f5", "#f7d4cf"], alpha=0.6)
        ax.contour(g1, g2, f, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"], linewidths=[1, 2, 1])
        idx = np.random.default_rng(self.random_state).permutation(len(y))[:max_points]
        for c, col, lab in [(0, "#1f77b4", "tidak batal (0)"), (1, "#d62728", "batal (1)")]:
            m = idx[y[idx] == c]
            ax.scatter(X[m, 0], X[m, 1], s=8, c=col, alpha=0.5, label=lab)
        sv = self.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=30, facecolors="none", edgecolors="k", linewidths=0.6,
                   label=f"support vector ({len(sv)})")
        ax.set(xlim=(g1.min(), g1.max()), ylim=(g2.min(), g2.max()), xlabel=feature_names[0],
               ylabel=feature_names[1], title=title or f"SVM {type(self.kernel).__name__}, C={self.C}")
        ax.legend(loc="upper right", fontsize=8)
        fig.tight_layout()
        fig.savefig(path, dpi=120)
        plt.close(fig)

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            return pickle.load(f)

### Scaling Strategy

Matriks kernel penuh untuk 21.856 baris latih bersih membutuhkan sekitar 3,8 GB dalam presisi ganda. Model menyimpan matriks kernel dari subsampel berstrata dan menggunakan paling banyak 5.000 baris latih di luar subsampel untuk kalibrasi Platt.

In [ ]:
def macro_f1(y, y_pred):
    return (f1(y, y_pred) + f1(1 - y, 1 - y_pred)) / 2

# one setting per kernel on the default 4,000-row subsample; GaussianKernel() uses gamma = 1/34 (about 0.029)
svm_kernels = {'linear': LinearKernel(), 'polynomial': PolynomialKernel(), 'gaussian': GaussianKernel()}
svm_C = {'linear': 1, 'polynomial': 1, 'gaussian': 10}
svm_models, svm_val_proba, svm_table = {}, {}, []
for name, kernel in svm_kernels.items():
    start = time.time()
    model = SVM(kernel=kernel, C=svm_C[name]).fit(X_train, y_train)
    svm_models[name] = model
    svm_val_proba[name] = model.predict_proba(X_val)
    svm_table.append({'kernel': name, 'C': model.C, 'support vectors': len(model.support_),
                      'converged': model.converged_, 'SMO iterations': model.n_iter_,
                      'seconds': time.time() - start, **scores(y_val, svm_val_proba[name]),
                      'macro_f1': macro_f1(y_val, (svm_val_proba[name] >= 0.5).astype(int))})
pd.DataFrame(svm_table).set_index('kernel').round(4)

In [ ]:
svm_model = SVM(GaussianKernel(gamma=0.03), C=10, max_samples=12000, max_iter=1_000_000)
start = time.time()
svm_model.fit(X_train, y_train)
svm_fit_seconds = time.time() - start
svm_final_proba = svm_model.predict_proba(X_val)
pd.Series({**scores(y_val, svm_final_proba), 'macro_f1': macro_f1(y_val, (svm_final_proba >= 0.5).astype(int)),
           'support vectors': len(svm_model.support_), 'converged': svm_model.converged_,
           'SMO iterations': svm_model.n_iter_, 'fit seconds': svm_fit_seconds}, name='validation')

### Insights

Pada subsampel 4.000 baris, kernel polinomial terpilih untuk CSV dengan akurasi 0,8113 dan F1 kelas batal 0,6787. Eksperimen Gaussian pada 12.000 baris menggunakan `C=10` dan `gamma=0.03`. Konfigurasi tambahan tersebut mencapai akurasi 0,8268, F1 kelas batal 0,7076, dan ROC-AUC 0,8784. Hasil kedua konfigurasi ditampilkan dengan nama yang berbeda pada tabel evaluasi.

### Decision Boundary (Bonus)

Visualisasi memakai `days_before_arrival` dan `nightly_rate_idr` yang telah dipraproses. Setiap kernel dilatih dengan `C=1` pada subsampel berstrata 2.000 baris. Garis utuh menunjukkan batas keputusan, garis putus-putus menunjukkan margin, dan lingkaran menandai vektor pendukung.

In [ ]:
svm_plot_cols = ['days_before_arrival', 'nightly_rate_idr']
for name, kernel in [('linear', LinearKernel()), ('polynomial', PolynomialKernel(degree=3)), ('gaussian', GaussianKernel())]:
    model = SVM(kernel, C=1, max_samples=2000, max_iter=1_000_000).fit(X_train[svm_plot_cols], y_train)
    path = CFG.OUTPUT_DIR / f'svm_boundary_{name}.png'
    model.plot_decision_boundary(X_train[svm_plot_cols], y_train, path, feature_names=svm_plot_cols)
    print(f'{name}: {len(model.support_vectors_)} support vectors, {model.n_iter_} iterations, converged={model.converged_}')
    display(Image(str(path)))

### Insights

Ketiga visualisasi menunjukkan tumpang tindih kelas pada bidang dua fitur tersebut. Jumlah vektor pendukung adalah 1.158 untuk kernel linear, 1.089 untuk polinomial, dan 1.063 untuk Gaussian. Model dengan semua fitur memakai 34 kolom masukan.

## Naive Bayes

Model mendiskretisasi fitur numerik tertentu, mengestimasi peluang dari frekuensi kategori, dan menggunakan penghalusan Laplace.

In [ ]:
"""Naive Bayes from scratch (NumPy): fitur kategorikal dan numerik berbin kuantil, smoothing Laplace.

Referensi: N. U. Maulidevi, slide "Bayes Net", IF3170 2024.
"""


class NaiveBayes:
    """alpha: smoothing Laplace. n_bins: jumlah bin kuantil untuk fitur numerik bernilai unik banyak.
    max_categorical: fitur numerik dengan nilai unik <= ini diperlakukan sebagai kategori."""

    def __init__(self, alpha=1.0, n_bins=20, max_categorical=10):
        self.alpha, self.n_bins, self.max_categorical = alpha, n_bins, max_categorical

    def fit(self, X, y):
        X = pd.DataFrame(X)
        y = np.asarray(y).astype(int)
        self.columns_ = list(X.columns)
        self.edges_, self.levels_, self.log_lik_ = {}, {}, {}

        n_class = np.bincount(y, minlength=2)
        self.log_prior_ = np.log((n_class + self.alpha) / (len(y) + 2 * self.alpha))

        for c in self.columns_:
            col = X[c]
            if pd.api.types.is_numeric_dtype(col) and col.nunique() > self.max_categorical:
                qs = np.linspace(0, 1, self.n_bins + 1)[1:-1]
                self.edges_[c] = np.unique(np.quantile(col.dropna().to_numpy(dtype=float), qs))
                self.levels_[c] = None
                n_levels = len(self.edges_[c]) + 1
            else:
                self.edges_[c] = None
                self.levels_[c] = sorted(col.dropna().unique())
                n_levels = len(self.levels_[c])

            codes = self._codes(c, col)
            counts = np.zeros((2, n_levels + 1))
            for k in (0, 1):
                seen = codes[(y == k) & (codes >= 0)]
                counts[k] = np.bincount(seen, minlength=n_levels + 1)
            self.log_lik_[c] = np.log((counts + self.alpha) / (counts.sum(1, keepdims=True) + self.alpha * (n_levels + 1)))
        return self

    def _codes(self, c, col):
        if self.edges_[c] is not None:
            v = col.to_numpy(dtype=float)
            codes = np.searchsorted(self.edges_[c], v, side="right")
            return np.where(np.isnan(v), -1, codes)
        codes = pd.Index(self.levels_[c]).get_indexer(col).astype(int)
        unseen = (codes == -1) & col.notna().to_numpy()
        return np.where(unseen, len(self.levels_[c]), codes)

    def predict_proba(self, X):
        X = pd.DataFrame(X)
        score = np.tile(self.log_prior_, (len(X), 1))
        for c in self.columns_:
            codes = self._codes(c, X[c])
            known = codes >= 0
            score[known] += self.log_lik_[c][:, codes[known]].T
        peak = score.max(axis=1, keepdims=True)
        log_norm = peak + np.log(np.exp(score - peak).sum(axis=1, keepdims=True))
        return np.exp(score - log_norm)[:, 1]

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            return pickle.load(f)

In [ ]:
nb_train, nb_val = split_train_val(train_df, seed=CFG.SEED)

def prepare_nb(train_part, val_part, steps=()):
    train_part, _ = drop_duplicates_and_invalid(train_part)
    pipeline = PreparationPipeline(list(steps)).fit(train_part)
    return (*pipeline.transform(train_part), *pipeline.transform(val_part))

nb_X_tr, nb_y_tr, nb_X_va, nb_y_va = prepare_nb(nb_train, nb_val)
nb_model = NaiveBayes(alpha=1.0, n_bins=20).fit(nb_X_tr, nb_y_tr)
nb_val_proba = nb_model.predict_proba(nb_X_va)

print(f'Train: {nb_X_tr.shape}  Validation: {nb_X_va.shape}')
print(f'Majority-class accuracy on validation: {1 - nb_y_va.mean():.4f}')
pd.Series(scores(nb_y_va, nb_val_proba), name='validation')

### Insights

Model dengan fitur asli, `alpha=1`, dan `n_bins=20` mencapai akurasi 0,7784 dan F1 kelas batal 0,6452 pada data validasi. Konfigurasi ini digunakan untuk CSV Naive Bayes.

### Cross-Validation and Feature Choice

Eksperimen lima lipatan membandingkan fitur asli dengan fitur asli yang ditambah fitur turunan. Pembagian berstrata memakai `seed=42`.

In [ ]:
nb_cv_raw = cross_validate(lambda: NaiveBayes(n_bins=20), prepare_nb, train_df)
nb_cv_derived = cross_validate(
    lambda: NaiveBayes(n_bins=20),
    lambda a, b: prepare_nb(a, b, steps=(construct_features,)),
    train_df,
)
pd.concat({'raw features': nb_cv_raw.loc['mean'], 'with derived features': nb_cv_derived.loc['mean']}, axis=1).T

### Insights

Pada validasi silang, fitur asli menghasilkan rata-rata akurasi 0,7812 dan F1 kelas batal 0,6462. Penambahan fitur turunan menaikkan recall dari 0,6099 menjadi 0,6342, tetapi akurasi turun menjadi 0,7677 dan F1 turun menjadi 0,6415. Konfigurasi pengumpulan mempertahankan fitur asli.

### Number of Bins

Fitur numerik dengan lebih dari sepuluh nilai unik dibagi berdasarkan kuantil. Tabel membandingkan 5, 10, 20, 30, dan 40 interval sebagai batas jumlah interval.

In [ ]:
nb_bins_grid = [5, 10, 20, 30, 40]
nb_bins_cv = pd.DataFrame({
    b: cross_validate(lambda b=b: NaiveBayes(n_bins=b), prepare_nb, train_df).loc['mean']
    for b in nb_bins_grid
}).T
nb_bins_cv.index.name = 'n_bins'
nb_bins_cv

### Insights

Di antara kandidat yang diuji, 40 interval memberi rata-rata F1 tertinggi, yaitu 0,6480. Konfigurasi 20 interval memperoleh F1 0,6462 dan tetap dipakai untuk mereproduksi CSV pengumpulan yang sudah ada.

## Model Evaluation

<!-- figure palette: #6d2fa3 -->

Kolom `submission_variant` menandai konfigurasi yang dipakai untuk menghasilkan CSV setelah pelatihan ulang. Tabel mencakup ketiga regularisasi regresi logistik, ketiga kernel SVM pada 4.000 baris, dan eksperimen Gaussian pada 12.000 baris. Variasi prapemrosesan KNN dan diskretisasi Naive Bayes ditampilkan pada bagian model masing-masing karena menggunakan skema percobaan tersendiri.

Di antara konfigurasi pengumpulan, KNN dan DTL mencatat akurasi tertinggi pada validasi lokal, masing-masing 0,8490 dan 0,8449. F1 kelas batal keduanya juga berdekatan, yaitu 0,7573 dan 0,7530.

In [ ]:
np.testing.assert_array_equal(X_val.index, T_val.index)
np.testing.assert_array_equal(X_val.index, nb_X_va.index)
np.testing.assert_array_equal(y_val, nb_y_va)

def best_by_accuracy(probabilities):
    return max(probabilities, key=lambda name: accuracy(y_val, probabilities[name] >= 0.5))


lr_submission_penalty = best_by_accuracy(lr_val_proba)
svm_submission_kernel = best_by_accuracy(svm_val_proba)

validation_models = {
    'knn': (knn_model, X_val, knn_val_proba, repr(knn_params)),
    'decision_tree': (dtl_model, T_val, dtl_val_proba, 'C4.5'),
    **{f'logreg_{name}': (model, X_val, lr_val_proba[name], f'penalty={name}')
       for name, model in lr_models.items()},
    **{f'svm_{name}': (model, X_val, svm_val_proba[name],
                       f'kernel={name}, C={model.C}, max_samples={model.max_samples}')
       for name, model in svm_models.items()},
    'svm_gaussian_12000': (svm_model, X_val, svm_final_proba,
                           'kernel=gaussian, gamma=0.03, C=10, max_samples=12000'),
    'naive_bayes': (nb_model, nb_X_va, nb_val_proba, 'alpha=1, n_bins=20'),
}

submission_models = (
    'knn', 'decision_tree', f'logreg_{lr_submission_penalty}',
    f'svm_{svm_submission_kernel}', 'naive_bayes',
)
saved_models = (*submission_models, 'svm_gaussian_12000')

validation_summary = pd.DataFrame.from_dict({
    name: {'configuration': configuration,
           'submission_variant': name in submission_models,
           **scores(y_val, probability),
           'macro_f1': macro_f1(y_val, (probability >= 0.5).astype(int))}
    for name, (_, _, probability, configuration) in validation_models.items()
}, orient='index').rename_axis('model')
validation_summary.round(4)

## Improvements (Optional)

Variasi prapemrosesan dan jarak KNN, kernel SVM, serta fitur dan jumlah interval Naive Bayes dicantumkan pada bagian model masing-masing.

In [ ]:
# Eksperimen tambahan ditampilkan pada bagian masing-masing model.

# Comparison with scikit-learn <a name="8"></a>

KNN dibandingkan pada parameter tetangga dan metrik jarak yang sama. Regresi logistik membandingkan ketiga regularisasi dengan menyesuaikan skala parameter penalti. Pembanding DTL memakai `DecisionTreeClassifier` dengan entropy dan `ccp_alpha=0.001`, serta kategori yang dikodekan dari data latih.

DTL C4.5 dan pembanding mencapai akurasi yang berdekatan (0,8449 dan 0,8446). Pembanding menghasilkan F1 kelas batal 0,7582, sedangkan C4.5 menghasilkan 0,7530. Algoritma split dan representasi kategori berbeda, sehingga perbandingan ini menilai hasil prediksi pada data yang sama.

In [ ]:
knn_sklearn = KNeighborsClassifier(n_neighbors=knn_params['k'], weights=knn_params['weights'],
                                   metric=knn_params['metric'], p=knn_params['p'], algorithm='brute')
compare_with_sklearn({'KNN': (KNN(**knn_params), knn_sklearn)}, X_train, y_train, X_val, y_val).round(4)

In [ ]:
# Decision Tree: scratch C4.5 vs DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.001)
dtl_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1,
                             encoded_missing_value=np.nan).fit(T_train[CATEGORICAL])


def encode_tree(T):
    T = T.copy()
    T[CATEGORICAL] = dtl_encoder.transform(T[CATEGORICAL])
    return T.astype(float)


dtl_sklearn = DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.001, random_state=CFG.SEED)
dtl_sklearn.fit(encode_tree(T_train), y_train)
pd.DataFrame({'scratch': scores(y_val, dtl_val_proba),
              'sklearn': scores(y_val, dtl_sklearn.predict_proba(encode_tree(T_val))[:, 1])}).T.round(4)

In [ ]:
# Logistic Regression: alpha = 1 / (C n), so both minimize the same objective
lr_comparison = {}
for penalty, model in lr_models.items():
    ratio = {'l1': 1.0, 'l2': 0.0, 'elasticnet': model.l1_ratio}[penalty]
    sk = SkLogisticRegression(C=1 / (model.alpha * len(X_train)), l1_ratio=ratio, solver='saga',
                              max_iter=20000, tol=1e-10).fit(X_train, y_train)
    lr_comparison[(penalty, 'scratch')] = scores(y_val, lr_val_proba[penalty])
    lr_comparison[(penalty, 'sklearn')] = {**scores(y_val, sk.predict_proba(X_val)[:, 1]),
                                           'max abs coef diff': np.abs(sk.coef_[0] - model.coef_).max()}
pd.DataFrame(lr_comparison).T.round(4)

### Support Vector Machine

Kedua pembanding memakai subsampel berstrata yang sama dengan model scratch. Pemeriksaan pertama membandingkan tanda fungsi keputusan untuk tiga kernel pada 4.000 baris. Pemeriksaan kedua membandingkan probabilitas Gaussian pada 12.000 baris. SVC melakukan kalibrasi internal, sementara model scratch memakai baris latih di luar subsampel.

In [ ]:
# SVM: SVC trained on the same stratified subsample the scratch SVM draws with its seed
X_val_np = X_val.to_numpy()
sklearn_kernels = {'linear': dict(kernel='linear'),
                   'polynomial': dict(kernel='poly', degree=3, gamma='auto', coef0=1.0),
                   'gaussian': dict(kernel='rbf', gamma='auto')}
svm_comparison = {}
for name, params in sklearn_kernels.items():
    model = svm_models[name]
    rows, _ = _stratified_subsample(y_train, model.max_samples, np.random.default_rng(model.random_state))
    sk = SVC(C=model.C, tol=model.tol, **params).fit(X_train.to_numpy()[rows], y_train[rows])
    ours, ref = model.decision_function(X_val_np), sk.decision_function(X_val_np)
    svm_comparison[name] = {
        'support vectors (scratch)': len(model.support_), 'support vectors (sklearn)': len(sk.support_),
        'same sign': np.mean(np.sign(ours) == np.sign(ref)),
        'accuracy (scratch)': accuracy(y_val, ours > 0), 'accuracy (sklearn)': accuracy(y_val, ref > 0),
        'roc_auc (scratch)': roc_auc(y_val, ours), 'roc_auc (sklearn)': roc_auc(y_val, ref),
    }
pd.DataFrame(svm_comparison).T.round(4)

In [ ]:
class SubsampledSVC:
    def __init__(self, max_samples, **params):
        self.max_samples, self.svc = max_samples, SVC(**params)

    def fit(self, X, y):
        idx, _ = _stratified_subsample(y, self.max_samples, np.random.default_rng(CFG.SEED))  # same rows as SVM(random_state=42)
        self.svc.fit(X.iloc[idx], y[idx])
        return self

    def predict_proba(self, X):
        return self.svc.predict_proba(X)

compare_with_sklearn(
    {'SVM': (SVM(GaussianKernel(gamma=0.03), C=10, max_samples=12000, max_iter=1_000_000),
             SubsampledSVC(12000, kernel='rbf', C=10, gamma=0.03, probability=True, random_state=CFG.SEED))},
    X_train, y_train, X_val, y_val,
)

### Insights

Dalam perbandingan tiga kernel, tanda fungsi keputusan scratch dan SVC cocok pada sedikitnya 99,89% baris validasi. Untuk Gaussian pada 12.000 baris, kedua implementasi menghasilkan ROC-AUC 0,8784 pada empat desimal. Akurasi dan F1 kelas batal sedikit berbeda setelah kalibrasi dan penerapan ambang 0,5.

### Naive Bayes

Pembanding menggunakan `CategoricalNB` dengan diskretisasi dan pengodean yang dipasang pada data latih. Uji pertama menyamakan kode kategori dan interval. Uji berikutnya membandingkan alur lengkap pada pembagian validasi dan pada lima lipatan.

In [ ]:
class SklearnCategoricalNB:
    def __init__(self, alpha=1.0, n_bins=20, max_categorical=10, class_prior=None):
        self.alpha, self.n_bins, self.max_categorical, self.class_prior = alpha, n_bins, max_categorical, class_prior

    def fit(self, X, y):
        X = pd.DataFrame(X)
        self.columns_ = list(X.columns)
        self.encoders_ = {}
        for c in self.columns_:
            col = X[c]
            if pd.api.types.is_numeric_dtype(col) and col.nunique() > self.max_categorical:
                enc = KBinsDiscretizer(self.n_bins, encode="ordinal", strategy="quantile")
                enc.fit(col.dropna().to_numpy(dtype=float).reshape(-1, 1))
            else:
                enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
                enc.fit(col.dropna().astype(object).to_numpy().reshape(-1, 1))
            self.encoders_[c] = enc
        self.model_ = CategoricalNB(alpha=self.alpha, class_prior=self.class_prior)
        self.model_.fit(self._codes(X), np.asarray(y).astype(int))
        return self

    def _codes(self, X):
        out = np.zeros((len(X), len(self.columns_)), dtype=int)
        for j, c in enumerate(self.columns_):
            col, enc = X[c], self.encoders_[c]
            ok = col.notna().to_numpy()
            if ok.any():
                v = col[ok].to_numpy(dtype=float if isinstance(enc, KBinsDiscretizer) else object).reshape(-1, 1)
                code = enc.transform(v).ravel().astype(int) + 1
                out[ok, j] = np.where(code < 0, 0, code)
        return out

    def predict_proba(self, X):
        return self.model_.predict_proba(self._codes(pd.DataFrame(X)))[:, 1]

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


def exact_equivalence(X_tr, y_tr, X_va, nb, n_bins=20):
    full_tr, full_va = ~X_tr.isna().any(axis=1).to_numpy(), ~X_va.isna().any(axis=1).to_numpy()
    X_tr, y_tr, X_va = X_tr[full_tr], np.asarray(y_tr)[full_tr], X_va[full_va]
    scratch = nb(n_bins=n_bins).fit(X_tr, y_tr)

    def codes(X):
        return np.column_stack([scratch._codes(c, X[c]) for c in scratch.columns_])

    sk = CategoricalNB(alpha=scratch.alpha, class_prior=np.exp(scratch.log_prior_),
                       min_categories=[scratch.log_lik_[c].shape[1] for c in scratch.columns_])
    sk.fit(codes(X_tr), np.asarray(y_tr).astype(int))
    p_s, p_k = scratch.predict_proba(X_va), sk.predict_proba(codes(X_va))[:, 1]
    return {"n_train": len(X_tr), "n_val": len(X_va), "max_abs_diff": float(np.abs(p_s - p_k).max()),
            "same_prediction": float(((p_s >= .5) == (p_k >= .5)).mean())}

In [ ]:
exact_equivalence(nb_X_tr, nb_y_tr, nb_X_va, NaiveBayes)

In [ ]:
compare_with_sklearn(
    {'Naive Bayes': (NaiveBayes(n_bins=20), SklearnCategoricalNB(n_bins=20))},
    nb_X_tr, nb_y_tr, nb_X_va, nb_y_va,
)

In [ ]:
nb_cv_sklearn = cross_validate(lambda: SklearnCategoricalNB(n_bins=20), prepare_nb, train_df)
pd.concat({'scratch': nb_cv_raw.loc['mean'], 'sklearn': nb_cv_sklearn.loc['mean']}, axis=1).T

### Insights

Pada validasi silang, model scratch dan pembanding memperoleh rata-rata akurasi 0,7812 dan 0,7810, serta F1 kelas batal 0,6462 dan 0,6459. Model scratch melewati atribut yang hilang, sedangkan alur pembanding mengodekannya sebagai kategori tersendiri. Uji dengan kode yang disamakan memisahkan pemeriksaan hitungan peluang dari perbedaan prapemrosesan.

# Saving and Loading Models <a name="9"></a>

<!-- artifacts: <model>_v3.pkl -->

Lima konfigurasi pengumpulan dan eksperimen Gaussian 12.000 disimpan dengan nama yang berbeda. Setiap model dimuat kembali dan diuji pada baris validasi yang sama. Pemeriksaan mengharuskan label persis sama dan probabilitas cocok dalam toleransi numerik.

In [ ]:
model_round_trip = []
for name in saved_models:
    model, features, probability, configuration = validation_models[name]
    path = CFG.OUTPUT_DIR / f'{name}_v3.pkl'
    model.save(path)
    restored = type(model).load(path)

    np.testing.assert_allclose(restored.predict_proba(features), probability,
                               rtol=1e-12, atol=1e-12, err_msg=name)
    np.testing.assert_array_equal(restored.predict(features), (probability >= 0.5).astype(int),
                                  err_msg=name)
    model_round_trip.append({'model': name, 'configuration': configuration,
                             'path': str(path), 'validation_rows': len(features),
                             'round_trip_passed': True})

pd.DataFrame(model_round_trip).set_index('model')

# Kaggle Submission <a name="10"></a>

Model dan alur prapemrosesan dilatih ulang menggunakan seluruh data latih setelah pembersihan. Data uji hanya ditransformasi. Setiap CSV berisi `id` dan `is_canceled`, dengan 6.844 baris prediksi pada dataset ini. `submission.csv` berasal dari KNN. Tabel keluaran mencantumkan konfigurasi dan nama CSV untuk setiap model. Pada pengumpulan yang dikonfirmasi oleh tim, CSV DTL memperoleh skor publik Kaggle 0,82005.

In [ ]:
"""Submission Kaggle: fit ulang pipeline dan model pada seluruh train.csv, test.csv hanya di-transform."""


def make_submission(model, pipeline, train_df, test_df, path, threshold=0.5):
    """Tulis CSV `id,is_canceled` berformat sample_submission.csv. Kembalikan DataFrame yang ditulis."""
    train_df, _ = drop_duplicates_and_invalid(train_df)
    pipeline.fit(train_df)
    X_tr, y_tr = pipeline.transform(train_df)
    X_te, _ = pipeline.transform(test_df)
    pred = model.fit(X_tr, y_tr).predict(X_te, threshold)
    sub = pd.DataFrame({ID_COL: test_df[ID_COL].to_numpy(), TARGET: pred})
    assert len(sub) == len(test_df) and sub[ID_COL].is_unique
    sub.to_csv(path, index=False)
    return sub


def check_reproducible(path_a, path_b):
    """True bila dua file submission berisi id dan prediksi yang sama (urutan baris bebas)."""
    a = pd.read_csv(path_a, dtype={ID_COL: str}).sort_values(ID_COL).reset_index(drop=True)
    b = pd.read_csv(path_b, dtype={ID_COL: str}).sort_values(ID_COL).reset_index(drop=True)
    return a.equals(b)

In [ ]:
submission_path = CFG.OUTPUT_DIR / 'submission.csv'
submission = make_submission(KNN(**knn_params), numeric_pipeline(), train_df, test_df, submission_path)

print(f'Rows: {len(submission)}  Share predicted canceled: {submission[TARGET].mean():.4f}')
reference = CFG.OUTPUT_DIR / 'submission_knn.csv'
if reference.exists():
    print('Identical to submission_knn.csv:', check_reproducible(submission_path, reference))
submission.head()

In [ ]:
per_model = {
    'decision_tree': (DecisionTree(categorical_features=tree_categorical), tree_pipeline()),
    f'logreg_{lr_submission_penalty}': (LogisticRegression(penalty=lr_submission_penalty),
                                        numeric_pipeline()),
    f'svm_{svm_submission_kernel}': (SVM(kernel=svm_kernels[svm_submission_kernel],
                                          C=svm_C[svm_submission_kernel]),
                                     numeric_pipeline()),
    'naive_bayes': (NaiveBayes(alpha=1.0, n_bins=20), PreparationPipeline([])),
}

model_submissions = {'knn': submission}
for name, (model, pipeline) in per_model.items():
    model_submissions[name] = make_submission(model, pipeline, train_df, test_df,
                                              CFG.OUTPUT_DIR / f'submission_{name}.csv')

submission_summary = pd.DataFrame.from_dict({
    name: {'file': 'submission.csv' if name == 'knn' else f'submission_{name}.csv',
           'configuration': validation_models[name][3],
           'validation_accuracy': validation_summary.loc[name, 'accuracy'],
           'share predicted canceled': sub[TARGET].mean(),
           'same prediction as KNN': (sub[TARGET].to_numpy() == submission[TARGET].to_numpy()).mean()}
    for name, sub in model_submissions.items()
}, orient='index').rename_axis('model')
submission_summary.round(4)

# Error Analysis <a name="11"></a>

FP adalah pemesanan yang diprediksi batal padahal tidak batal. FN adalah pembatalan aktual yang tidak terdeteksi. Tingkat FP dihitung terhadap jumlah kelas negatif aktual dan tingkat FN terhadap jumlah kelas positif aktual.

Pada eksekusi lokal, DTL menghasilkan 349 FP dan 500 FN. Pada pemesanan dengan jarak paling lama tujuh hari sebelum kedatangan, model melewatkan 43 dari 69 pembatalan (62,3%). Pada rentang lebih dari 90 hari, model melewatkan 151 dari 1.072 pembatalan (14,1%). Kedua kelompok berukuran berbeda. Temuan ini menggambarkan pola pada data validasi yang digunakan.

In [ ]:
error_summary = []
for name, (_, _, probability, _) in validation_models.items():
    prediction = (probability >= 0.5).astype(int)
    (tn, fp), (fn, tp) = confusion_matrix(y_val, prediction)
    error_summary.append({'model': name, 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
                          'false_positive_rate': fp / (tn + fp) if tn + fp else np.nan,
                          'false_negative_rate': fn / (fn + tp) if fn + tp else np.nan})

display(pd.DataFrame(error_summary).set_index('model').round(4))

dtl_pred = (dtl_val_proba >= 0.5).astype(int)
dtl_errors = pd.DataFrame({
    'actual_negative': y_val == 0,
    'actual_positive': y_val == 1,
    'FP': (y_val == 0) & (dtl_pred == 1),
    'FN': (y_val == 1) & (dtl_pred == 0),
}, index=T_val.index)

dtl_groups = {column: T_val[column] for column in CATEGORICAL}
dtl_groups['days_before_arrival'] = pd.cut(T_val['days_before_arrival'],
                                         bins=[-np.inf, 7, 30, 90, np.inf])
dtl_error_by_feature = {}
for feature, groups in dtl_groups.items():
    groups = groups.astype(object).where(groups.notna(), '<missing>')
    grouped = dtl_errors.groupby(groups, sort=False)
    summary = grouped.sum().assign(rows=grouped.size())
    summary['false_positive_rate'] = summary['FP'] / summary['actual_negative'].replace(0, np.nan)
    summary['false_negative_rate'] = summary['FN'] / summary['actual_positive'].replace(0, np.nan)
    dtl_error_by_feature[feature] = summary

pd.concat(dtl_error_by_feature, names=['feature', 'group']).round(4)